### The purpose of this notebook is to recreate the baseline visual instruction tuning pipeline in the LLaVA paper

In [ ]:
import importlib.util

if importlib.util.find_spec('tensorflow') is None:
  print("Installing required packages...")
  %pip install -q dotenv
  %pip install -q kagglehub
  %pip install -q ipywidgets
  %pip install -q tensorflow
  %pip install -q tensorflow_datasets
  %pip install -q tensorboardX
  %pip install -q transformers
  %pip install -q grain
  %pip install -q git+https:/github.com/jax-ml/jax
  %pip install git+https:/github.com/google/tunix
  %pip install git+https:/github.com/google/qwix
  %pip uninstall -q flax -y
  %pip install git+https:/github.com/google/flax
  %pip install -q huggingface_hub
  %pip install -q datasets
  %pip install -q 'numpy>2'

## This is how our baseline workload looks like



## Project Structure

### Please create such directories

In [ ]:
import os
from pathlib import Path

# Get the directory where the script is located
#TODO:
SCRIPT_DIR = "../scripts"

# The project root is one level up from 'scripts/'
PROJECT_ROOT = Path(SCRIPT_DIR).parent

# Define your data paths relative to the project root
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
EMBEDDINGS_DIR = PROCESSED_DATA_DIR / "clip_embeddings"

# Ensure directories exist
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")

In [ ]:
#UTILITY FUNCTIONS
def show_hbm_usage():
  """Displays memory usage per device."""
  fmt_size = functools.partial(humanize.naturalsize, binary=True)

  for d in jax.local_devices():
    stats = d.memory_stats()
    used = stats["bytes_in_use"]
    limit = stats["bytes_limit"]
    print(f"Using {fmt_size(used)} / {fmt_size(limit)} ({used/limit:%}) on {d}")

## Our Datasets 

### For baseline, we are using COCO for both stage 1 and stage 2, instead of CC3M and LLaVA-instruct-158k, for simplicity and debugging

In [ ]:
%cd /home/CS159FinalProject-main
!pwd

In [ ]:
!python3 scripts/prepare_stage1_dataset.py
!python3 scripts/convert_alignment_format.py

In [ ]:
# Download the COCO 2017 dataset and annotations 
!mkdir -p /home/CS159FinalProject/data/raw
%cd /home/CS159FinalProject/data/raw
!pwd

!wget -c http:/images.cocodataset.org/zips/train2017.zip
!wget -c http:/images.cocodataset.org/annotations/annotations_trainval2017.zip

!unzip -n train2017.zip
!unzip -n annotations_trainval2017.zip

In [ ]:
# Match the COCO captions with the corresponding images and save in a new JSON file for stage 1 training.
!python ../scripts/prepare_stage1_dataset.py

In [ ]:
# We are adding a simple instruction to the captions to make them more suitable for training the projector in stage 1.
# Make sure to execute this at project root directory (CS159FinalProject/) since the paths in the script are relative to it.
%cd /home/CS159FinalProject-main
!python3 scripts/convert_alignment_format.py

In [ ]:
# Loggin to HF

import os

import json
import jax
import jax.numpy as jnp
import numpy as np

from PIL import Image
import os
import kagglehub

try:
  from google.colab import userdata
  USE_COLAB = True

  %pip uninstall -q wandb -y  # wandb is glitchy with tunix in colab

#TODO: Optional HF token and kaggle credentials for colab. If not set, will skip login and rely on kagglehub to download datasets.
  os.environ["TOKEN"] = "HF_TOKEN"
except:
  USE_COLAB = False

  from dotenv import load_dotenv
  load_dotenv()
  print("Using env vars to login")

  import nest_asyncio
  nest_asyncio.apply()
  print("nest_asyncio applied")

  # Only using wandb on TPU VM because it has strange bugs on Colab
  !pip install -q wandb
  import wandb
  # Check if WANDB_API_KEY is set before logging in
  if "WANDB_API_KEY" in os.environ and os.environ["WANDB_API_KEY"]:
      wandb.login(key=os.environ["WANDB_API_KEY"])
  else:
      print("WANDB_API_KEY not found. Skipping wandb login.")

if "KAGGLE_USERNAME" not in os.environ or "KAGGLE_KEY" not in os.environ:
  kagglehub.login()

if "TOKEN" in os.environ and os.environ["TOKEN"]:
    TOKEN = os.environ["TOKEN"]
    !hf auth login --token "$TOKEN"
else:
    print("TOKEN not found. Skipping Hugging Face login.")

In [ ]:
# ====== Sharding ======
# Adjust mesh based on your TPU memory and model size.
NUM_TPUS = len(jax.devices())
if NUM_TPUS == 8:
  MESH_COUNTS = (2, 4)
elif NUM_TPUS == 4:
  MESH_COUNTS = (1, 4)
elif NUM_TPUS == 1:
  MESH_COUNTS = (1, 1)
else:
  raise ValueError(f"Unsupported number of TPUs: {NUM_TPUS}")

MESH = [
    MESH_COUNTS,
    ("fsdp", "tp"),
]

# Get HF access to llama 3.2 1B Instruct model files

https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

In [ ]:
!find /home/CS159FinalProject-main -name model.py -o -name params.py

In [ ]:
import sys
print(sys.executable)
print(sys.version)

import jax
print(jax.devices())

In [ ]:
%pip uninstall -y flax
%pip install -q "flax==0.8.5"
# We need to bypass the standard transformer loader as it would have a stuck progress bar on TPU. Setting this environment variable disables the progress bar and allows the model to load successfully.
# First let's download Llama
import sys

sys.path.insert(0, "../scripts")

import model as llama_model
import params as llama_params

print("imports ok")



In [ ]:
# Export HF API Key to get access to the model files. Make sure to set TOKEN in your environment variables before running this cell.

#TODO: Again, set HF API KEY
os.environ["TOKEN"] = "HF_TOKEN"

In [ ]:
! HF_HUB_ENABLE_HF_TRANSFER

!hf download meta-llama/Llama-3.2-1B-Instruct \
  --include "config.json" "generation_config.json" \
            "tokenizer.json" "tokenizer_config.json" \
            "special_tokens_map.json" \
            "model.safetensors" "model.safetensors.index.json" \
  --local-dir CS159FinalProject/temp/hf_models/Llama-3.2-1B-Instruct

In [ ]:
import jax
import flax
from flax import nnx

print(jax.__version__)
print(flax.__version__)
print("nnx ok")

In [ ]:
import sys
sys.path.insert(0, "../scripts")

import model as llama_model
import params as llama_params

print("imports ok")

In [ ]:
# Choose the correct config for the model you downloaded. This should match the config.json file in the model directory.
# Config is used to define the architecture of the model and must match the parameters used during training. 
# If you downloaded a different model, make sure to update this config accordingly.
cfg = llama_model.ModelConfig.llama3p2_1b_instruct()

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

MESH = ((1, 8), ('fsdp', 'tp'))

In [ ]:
import jax
print(jax.devices())
print("device count:", len(jax.devices()))
print("default backend:", jax.default_backend())

In [ ]:
from jax.sharding import Mesh
from jax.experimental import mesh_utils
import jax
import jax.numpy as jnp

llama_dir = "../models/llama"

mesh_shape = MESH[0]     
mesh_axis_names = MESH[1]   

device_mesh = mesh_utils.create_device_mesh(mesh_shape)
mesh = Mesh(device_mesh, axis_names=mesh_axis_names)

llama3_1b = llama_params.create_model_from_safe_tensors(
    file_dir=llama_dir,
    config=cfg,
    mesh=mesh,
    dtype=jnp.bfloat16,
)

In [ ]:
# WE can display the model architecture using nnx.display. This will show the layers and parameters of the model, which can be useful for debugging and understanding the model structure.
from flax import nnx
nnx.display(llama3_1b)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    llama_dir,
    local_files_only=True,
)

download CLIP locally
→ load FlaxCLIPModel from local files
→ keep only vision tower for feature extraction
→ return processor + model + vision hidden size

In [ ]:
# Now the same deal for CLIP. We need the vision encoder to extract image features for the projector training in stage 1, and also for the multimodal finetuning in stage 2.
from clip_helpers import download_clip_flax

clip_dir = download_clip_flax(
    local_dir="../home/clip-vit-base-patch32"
)
print(clip_dir)

In [ ]:
from clip_helpers import build_clip_vision_tower

clip_bundle = build_clip_vision_tower()
print("CLIP hidden size:", clip_bundle.hidden_size)
print("Image size:", clip_bundle.image_size)
print("Patch size:", clip_bundle.patch_size)

In [ ]:
#Sanity Check
from clip_helpers import load_clip_flax_local
clip_bundle = load_clip_flax_local()
print(type(clip_bundle.model))

In [ ]:
model = clip_bundle.model

### Sanity Check
from PIL import Image
import requests
from transformers import AutoProcessor, FlaxCLIPModel

model = FlaxCLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(images=image, return_tensors="np")

image_features = model.get_image_features(**inputs)

In [ ]:
import jax

In [ ]:
# Check available JAX devices, essential for setting up mesh geometry
jax.devices()

In [ ]:
from flax import nnx
import jax
import jax.numpy as jnp


class VisionProjector(nnx.Module):
    def __init__(self, in_dim: int=768, out_dim: int = 2048, *, rngs: nnx.Rngs):
        self.proj = nnx.Linear(
            in_features=in_dim,
            out_features=out_dim,
            use_bias=False,
            rngs=rngs,
        )

    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        # x: [B, N_vis, D_clip]
        return self.proj(x)  # [B, N_vis, D_llama]
    
    
def serialize_stage1_sample(tokenizer, sample, max_len=128):
    instruction = sample["instruction"]
    response = sample["response"]

    prefix = f"USER: {instruction}\nASSISTANT:"
    full_text = prefix + " " + response

    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
    prefix_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]

    full_ids = full_ids[:max_len]
    labels = full_ids.copy()

    prefix_len = min(len(prefix_ids), len(labels))
    labels[:prefix_len] = [-100] * prefix_len

    return {
        "image": sample["image"],
        "input_ids": full_ids,
        "labels": labels,
    }

In [ ]:
def masked_cross_entropy_loss(logits, labels):
    # logits: [B, L, V]
    # labels: [B, L]
    #B: Batch size, L: Sequence length, V: Vocabulary size
    shift_logits = logits[:, :-1, :] #Select the logits for all tokens except the last one, since we are predicting the next token at each position.
    shift_labels = labels[:, 1:] #Shift the labels to the left by one position, so that each position's label corresponds to the next token that the model should predict.

    valid = shift_labels != -100
    safe_labels = jnp.where(valid, shift_labels, 0)

    log_probs = jax.nn.log_softmax(shift_logits, axis=-1)
    token_logp = jnp.take_along_axis(
        log_probs,
        safe_labels[..., None],
        axis=-1
    ).squeeze(-1)

    loss = -jnp.sum(token_logp * valid) / jnp.maximum(jnp.sum(valid), 1)
    return loss

In [ ]:
# During training, confirm these dimensions: 

#vision_feats: [B, N_vis, D_clip]
#vis_embeds: [B, N_vis, 2048]
#text_embeds: [B, T, 2048]
#input_embeds: [B, N_vis + T, 2048]
#logits: [B, N_vis + T, vocab_size]

## Stage 1 Orchestration

Now let’s wire the whole thing.

The best first orchestration is:

1. load processed Stage 1 JSON
2. tokenize text examples
3. precompute CLIP vision features
4. freeze CLIP + Llama
5. train projector only

In [ ]:
#Compute stage 1 tokenized dataset

import json
import os

def build_tokenized_stage1_dataset(tokenizer, input_json, output_json, max_len=128):
    with open(input_json, "r") as f:
        data = json.load(f)

    processed = []
    for sample in data:
        ex = serialize_stage1_sample(tokenizer, sample, max_len=max_len)
        processed.append(ex)

    os.makedirs(os.path.dirname(output_json), exist_ok=True)
    with open(output_json, "w") as f:
        json.dump(processed, f)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "../models/llama"
)

print(type(tokenizer))
print("tokenizer ok")

In [ ]:
#Build tokenized dataset for stage 1 training. This will create a new JSON file where each sample contains the tokenized input IDs and labels, ready for training the vision projector. Make sure to run this after downloading the COCO dataset and preparing the alignment JSON.

#TODO: Modify this accordingly
# 1. Define Paths (Ensuring consistency with your directory structure)
ALIGNMENT_JSON = "../data/processed/stage1_alignment/alignment_chat.json"
TOKENIZED_JSON = "../data/processed/stage1_alignment/alignment_tokenized.json"
IMAGE_ROOT = "../train2017" # Adjust if your images are in 'images/'
FEATURE_DIR = "../data/processed/clip_embeddings"
MANIFEST_JSON = "../data/processed/stage1_alignment/stage1_manifest.json"

#Uncomment this block if you need to re-run the tokenization step. Make sure to run this after preparing the alignment JSON.
#--- STEP 1: Tokenize the text ---
print("Step 1: Tokenizing dataset...")
build_tokenized_stage1_dataset(
    tokenizer=tokenizer,
    input_json=ALIGNMENT_JSON,
    output_json=TOKENIZED_JSON,
    max_len=128
)


In [ ]:
def make_clip_feature_fn(clip_bundle):
    
    '''Returns a function that takes in raw pixel values and returns the CLIP vision features. 
    This function is JIT-compiled for efficiency.'''
    
    params = clip_bundle.model.params

    @jax.jit
    def get_features(pixel_values):
        outputs = clip_bundle.model(
            pixel_values=pixel_values,
            params=params,
            output_hidden_states=True,
        )
        return outputs.hidden_states[-2]

    return get_features


In [ ]:
import json
import os
import numpy as np
import jax.numpy as jnp
from PIL import Image


def precompute_clip_features_jitted(
    clip_bundle,
    tokenized_json,
    image_root,
    output_dir,
):
    with open(tokenized_json, "r") as f:
        data = json.load(f)

    os.makedirs(output_dir, exist_ok=True)

    get_features_compiled = make_clip_feature_fn(clip_bundle) #Get the JIT-compiled function for feature extraction

    print(f"Extracting features for {len(data)} images...")

    for i, sample in enumerate(data):
        try:
            image_path = os.path.join(image_root, sample["image"])
            img = Image.open(image_path).convert("RGB")

            clip_inputs = clip_bundle.processor(
                images=img,
                return_tensors="np",
            )

            pixel_values = jnp.array(clip_inputs["pixel_values"])

            # [B, seq_len, hidden_dim]
            hidden_states_penultimate = get_features_compiled(pixel_values) #Extract features using the JIT-compiled function

            # save single example: [seq_len, hidden_dim]
            vision_feats = np.array(hidden_states_penultimate[0])

            save_path = os.path.join(
                output_dir,
                sample["image"].replace(".jpg", ".npy"),
            )
            np.save(save_path, vision_feats)

            if i % 100 == 0:
                print(f"Processed {i}/{len(data)}...")

        except Exception as e:
            print(f"Error on {sample['image']}: {e}")
            
# Precompute CLIP features for all images in the tokenized dataset. This will save the features as .npy files in the specified output directory. Make sure to run this after tokenizing the dataset and before training the projector, as the training loop will load these precomputed features.
precompute_clip_features_jitted(
    clip_bundle=clip_bundle,
    tokenized_json=TOKENIZED_JSON,
    image_root=IMAGE_ROOT,
    output_dir=FEATURE_DIR,
)   

In [ ]:
#Double check that we are using FlaxCLIPVISIONModel
print(type(clip_bundle.model))
print(clip_bundle.model.__class__.__name__)
print(type(clip_bundle.model.config))
print(clip_bundle.model.config.model_type)

In [ ]:
# 3. Build trainable examples
#Build the manifest for stage 1 training. This manifest will be used by the training loop to load the precomputed CLIP features and the tokenized text inputs for each training example.

from pathlib import Path
from build_stage1_manifest import build_stage1_manifest

print("\nStep 3: Building final training manifest...")
build_stage1_manifest(
    tokenized_json=TOKENIZED_JSON,
    feature_dir=FEATURE_DIR,
    output_json=Path(MANIFEST_JSON)
)

print(f"\nSuccess! Manifest created at: {MANIFEST_JSON}")

In [ ]:
# Check manifest is valid
with open(MANIFEST_JSON, "r") as f:
    sample_manifest = json.load(f)[0]
    
print("Manifest Sample Check:")
print(f" - Vision Path: {sample_manifest['vision_path']}")
print(f" - Vision Exists: {os.path.exists(sample_manifest['vision_path'])}")
print(f" - Token Count: {len(sample_manifest['input_ids'])}")

In [ ]:
# 4 Collator

import numpy as np
import ml_dtypes


def pad_list(x, length, pad_value):
    return x + [pad_value] * (length - len(x))


def stage1_collate_fn(batch):
    max_text_len = max(len(x["input_ids"]) for x in batch)

    vision_feats = []
    input_ids = []
    labels = []

    for sample in batch:
        vf = np.load(sample["vision_path"])   # [50, 768]

        if vf.dtype.kind == "V":
            vf = vf.view(ml_dtypes.bfloat16).astype(np.float32)
        else:
            vf = vf.astype(np.float32)

        vision_feats.append(vf)
        input_ids.append(pad_list(sample["input_ids"], max_text_len, 0))
        labels.append(pad_list(sample["labels"], max_text_len, -100))

    vision_feats = np.stack(vision_feats, axis=0)
    input_ids = np.array(input_ids, dtype=np.int32)
    labels = np.array(labels, dtype=np.int32)

    return {
        "vision_feats": vision_feats,
        "input_ids": input_ids,
        "labels": labels,
    }

In [ ]:
#5 Build multimodal batch inside training loop, since we need to feed the raw images to the CLIP vision tower and get the projected embeddings on the fly for each batch. 

import jax.numpy as jnp


def make_multimodal_inputs(llama_model, projector, batch):
    vision_feats = jnp.array(batch["vision_feats"])
    input_ids = jnp.array(batch["input_ids"])
    labels = jnp.array(batch["labels"])

    vis_embeds = projector(vision_feats)                # [B, N_vis, D]
    txt_embeds = llama_model.embedder.encode(input_ids) # [B, T, D]

    input_embeds = jnp.concatenate([vis_embeds, txt_embeds], axis=1)

    bsz, n_vis, _ = vis_embeds.shape
    _, tlen = input_ids.shape
    seq_len = n_vis + tlen

    vis_labels = -100 * jnp.ones((bsz, n_vis), dtype=jnp.int32)
    full_labels = jnp.concatenate([vis_labels, labels], axis=1)

    positions = jnp.broadcast_to(
        jnp.arange(seq_len)[None, :],
        (bsz, seq_len)
    )

    attention_mask = jnp.tril(
        jnp.ones((seq_len, seq_len), dtype=bool)
    )
    attention_mask = jnp.broadcast_to(
        attention_mask[None, :, :],
        (bsz, seq_len, seq_len)
    )

    return {
        "input_embeds": input_embeds,
        "labels": full_labels,
        "positions": positions,
        "attention_mask": attention_mask,
    }

In [ ]:
# 6 Optimizer and training step
import optax
from flax import nnx
import jax
#Freeze the CLIP vision tower and LLaMA model during stage 1 training, only train the projector.

tx = optax.adamw(learning_rate=1e-3, weight_decay=0.0)

projector = VisionProjector(
    in_dim=clip_bundle.hidden_size,
    out_dim=llama3_1b.config.embed_dim,
    rngs=nnx.Rngs(0),
)

projector_graphdef, projector_state = nnx.split(projector) #Split the projector module into a graph definition (which contains the structure of the module) and a state (which contains the parameters). This allows us to merge the state back into the module during the training step, which is necessary for JIT compilation.
opt_state = tx.init(projector_state)

In [ ]:
import jax
import jax.numpy as jnp

def llama_forward_from_embeddings(llama_model, input_embeds, positions, attention_mask):
    x = input_embeds
    cache = None

    for layer in llama_model.layers:
        cache, x = layer(
            x=x,
            segment_pos=positions,
            cache=cache,
            attn_mask=attention_mask,
        )

    x = llama_model.final_norm(x)

    if hasattr(llama_model.embedder, "decode"):
        logits = llama_model.embedder.decode(x)
    else:
        raise AttributeError("llama_model.embedder has no decode() method.")

    return logits, cache

In [ ]:
@nnx.jit # Use nnx.jit for compatibility with nnx modules
def train_step(projector_state, opt_state, batch, projector_graphdef, llama_state, llama_graphdef):
    def loss_fn(proj_state):
        # Merge states back into modules inside the loss function
        projector = nnx.merge(projector_graphdef, proj_state)
        llama_model = nnx.merge(llama_graphdef, llama_state)

        mm = make_multimodal_inputs(llama_model, projector, batch)

        logits, _ = llama_forward_from_embeddings(
            llama_model,
            input_embeds=mm["input_embeds"],
            positions=mm["positions"],
            attention_mask=mm["attention_mask"],
        )

        return masked_cross_entropy_loss(logits, mm["labels"])

    loss, grads = jax.value_and_grad(loss_fn)(projector_state)
    updates, opt_state = tx.update(grads, opt_state, projector_state)
    projector_state = optax.apply_updates(projector_state, updates)
    return projector_state, opt_state, loss

In [ ]:
import random
import json
import jax.numpy as jnp

# split the LLM once so stage 1 can use the frozen state correctly
llama_graphdef, llama_state = nnx.split(llama3_1b)

def iterate_minibatches(data, batch_size):
    idxs = list(range(len(data)))
    random.shuffle(idxs)

    for i in range(0, len(idxs), batch_size):
        batch = [data[j] for j in idxs[i:i + batch_size]]
        yield stage1_collate_fn(batch)


def run_stage1_training(
    manifest_json,
    num_epochs=1,
    batch_size=2,
):
    global projector_state, opt_state, llama_graphdef, llama_state

    with open(manifest_json, "r") as f:
        manifest = json.load(f)

    step = 0
    for epoch in range(num_epochs):
        for batch in iterate_minibatches(manifest, batch_size=batch_size):
            batch = {
                "vision_feats": jnp.asarray(batch["vision_feats"], dtype=jnp.float32),
                "input_ids": jnp.asarray(batch["input_ids"], dtype=jnp.int32),
                "labels": jnp.asarray(batch["labels"], dtype=jnp.int32),
            }

            projector_state, opt_state, loss = train_step(
                projector_state,
                opt_state,
                batch,
                projector_graphdef,
                llama_state,
                llama_graphdef,
            )

            if step % 10 == 0:
                print(f"epoch={epoch} step={step} loss={float(loss):.4f}")
            step += 1

In [ ]:
run_stage1_training(
    manifest_json=MANIFEST_JSON,
    num_epochs=1,
    batch_size=4,
)

In [ ]:
import os
import pickle

PROJECTOR_STATE_PATH = "../train/checkpoints/projector_stage1.pkl"
os.makedirs(os.path.dirname(PROJECTOR_STATE_PATH), exist_ok=True)

with open(PROJECTOR_STATE_PATH, "wb") as f:
    pickle.dump(projector_state, f)

print("Saved stage-1 projector to:", PROJECTOR_STATE_PATH)

## Step 2: Multimodal Finetuning


1. initialize projector from Stage 1
2. keep CLIP frozen
3. keep projector trainable
4. unfreeze Llama
5. train on image + instruction + response instead of simple caption prompts

In [ ]:
from pathlib import Path
import json
from collections import Counter

STAGE2_ROOT = Path("../data/processed/stage2_instruction")

BASELINE_VARIANT = "llama"
QUALITY_VARIANTS = ["qwen", "llama"]   # use the exact folder name here
ALL_VARIANTS = [BASELINE_VARIANT] + QUALITY_VARIANTS

if len(ALL_VARIANTS) != len(set(ALL_VARIANTS)):
    raise ValueError(f"Duplicate names found in ALL_VARIANTS: {ALL_VARIANTS}")

required_base = {"image", "image_id", "generator_model", "task_type", "instruction", "response"}
allowed_tasks = {"conversation", "detailed_description", "complex_reasoning"}

variant_image_sets = {}
variant_task_counts = {}
variant_row_counts = {}

print("Baseline variant:", BASELINE_VARIANT)
print("Quality variants:", QUALITY_VARIANTS if QUALITY_VARIANTS else "[none yet]")
print("All variants:", ALL_VARIANTS)

for variant in ALL_VARIANTS:
    path = STAGE2_ROOT / variant / "stage2_dataset.jsonl"
    print(f"\n=== {variant} ===")
    print("Path:", path)

    if not path.exists():
        print("MISSING FILE")
        continue

    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    print("Rows:", len(rows))
    if not rows:
        continue

    task_counts = Counter(r.get("task_type", "UNKNOWN") for r in rows)
    print("Task counts:", dict(task_counts))

    images = {r["image_id"] for r in rows if "image_id" in r}
    print("Unique image_ids:", len(images))

    bad_rows = []
    for i, r in enumerate(rows):
        missing = required_base - set(r.keys())
        if missing:
            bad_rows.append((i, f"missing keys: {missing}"))
            continue

        if r["task_type"] not in allowed_tasks:
            bad_rows.append((i, f"bad task_type: {r['task_type']}"))

        if r["task_type"] == "conversation":
            if "history" not in r or "turn_index" not in r:
                bad_rows.append((i, "conversation row missing history or turn_index"))

    print("Bad rows found:", len(bad_rows))
    if bad_rows[:5]:
        print("First few bad rows:", bad_rows[:5])

    print("Sample keys:", sorted(rows[0].keys()))

    variant_image_sets[variant] = images
    variant_task_counts[variant] = task_counts
    variant_row_counts[variant] = len(rows)

if BASELINE_VARIANT in variant_image_sets and QUALITY_VARIANTS:
    print("\n=== Image-set consistency against baseline ===")
    base_images = variant_image_sets[BASELINE_VARIANT]

    for variant in QUALITY_VARIANTS:
        if variant not in variant_image_sets:
            print(f"{variant}: skipped (dataset file missing)")
            continue

        same_images = (variant_image_sets[variant] == base_images)
        print(f"{variant}: same image_id set as baseline -> {same_images}")

        if not same_images:
            only_in_baseline = sorted(base_images - variant_image_sets[variant])[:10]
            only_in_variant = sorted(variant_image_sets[variant] - base_images)[:10]
            print("  only in baseline:", only_in_baseline)
            print("  only in variant:", only_in_variant)
else:
    print("\nNo quality variants registered yet, or baseline file not found.")

In [ ]:
POOL_REFERENCE_VARIANT = "gemma"   # temporary reference for now
QUALITY_IMAGE_COUNT = 5000
VAL_IMAGE_COUNT = 1000
SPLIT_SEED = 42

In [ ]:
from pathlib import Path
import json
from collections import Counter

STAGE2_ROOT = Path("../data/processed/stage2_instruction")
DATA_VARIANT = "llama"   # dataset actually used in our experiments

required_keys = {"image", "image_id", "generator_model", "task_type", "instruction", "response"}
allowed_tasks = {"conversation", "detailed_description", "complex_reasoning"}

path = STAGE2_ROOT / DATA_VARIANT / "stage2_dataset.jsonl"
print("Dataset path:", path)

if not path.exists():
    raise FileNotFoundError(f"Missing dataset file: {path}")

rows = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print("Total rows:", len(rows))

task_counts = Counter(r.get("task_type", "UNKNOWN") for r in rows)
print("Task counts:", dict(task_counts))

images = {r["image_id"] for r in rows if "image_id" in r}
print("Unique image_ids:", len(images))

bad_rows = []
for i, r in enumerate(rows):
    missing = required_keys - set(r.keys())
    if missing:
        bad_rows.append((i, f"missing keys: {missing}"))
        continue

    if r["task_type"] not in allowed_tasks:
        bad_rows.append((i, f"bad task_type: {r['task_type']}"))

    if r["task_type"] == "conversation":
        if "history" not in r or "turn_index" not in r:
            bad_rows.append((i, "conversation row missing history or turn_index"))

print("Bad rows found:", len(bad_rows))
if bad_rows[:5]:
    print("First few bad rows:", bad_rows[:5])

print("Sample keys:", sorted(rows[0].keys()))

In [ ]:
from pathlib import Path
import json
import random

STAGE2_ROOT = Path("../data/processed/stage2_instruction")
DATA_VARIANT = "llama"   # the dataset we actually used

dataset_path = STAGE2_ROOT / DATA_VARIANT / "stage2_dataset.jsonl"
if not dataset_path.exists():
    raise FileNotFoundError(f"Missing dataset file: {dataset_path}")

image_ids = set()
with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        image_ids.add(row["image_id"])

image_ids = sorted(image_ids)
print("Total unique images:", len(image_ids))

if len(image_ids) < QUALITY_IMAGE_COUNT:
    raise ValueError(
        f"Only {len(image_ids)} images found, but QUALITY_IMAGE_COUNT={QUALITY_IMAGE_COUNT}."
    )

rng = random.Random(SPLIT_SEED)
selected_image_ids = image_ids[:]
rng.shuffle(selected_image_ids)
selected_image_ids = selected_image_ids[:QUALITY_IMAGE_COUNT]

if VAL_IMAGE_COUNT >= len(selected_image_ids):
    raise ValueError(
        f"VAL_IMAGE_COUNT={VAL_IMAGE_COUNT} is too large for only "
        f"{len(selected_image_ids)} selected images."
    )

val_image_ids = set(selected_image_ids[:VAL_IMAGE_COUNT])
train_image_ids = set(selected_image_ids[VAL_IMAGE_COUNT:])

pool_info = {
    "data_variant": DATA_VARIANT,
    "quality_image_count": QUALITY_IMAGE_COUNT,
    "selected_image_ids": sorted(selected_image_ids),
}

split_info = {
    "seed": SPLIT_SEED,
    "data_variant": DATA_VARIANT,
    "quality_image_count": QUALITY_IMAGE_COUNT,
    "num_train_images": len(train_image_ids),
    "num_val_images": len(val_image_ids),
    "train_image_ids": sorted(train_image_ids),
    "val_image_ids": sorted(val_image_ids),
}

pool_path = STAGE2_ROOT / "selected_quality_pool.json"
split_path = STAGE2_ROOT / "train_val_split.json"

with open(pool_path, "w", encoding="utf-8") as f:
    json.dump(pool_info, f, ensure_ascii=False, indent=2)

with open(split_path, "w", encoding="utf-8") as f:
    json.dump(split_info, f, ensure_ascii=False, indent=2)

print("\nSaved:", pool_path)
print("Saved:", split_path)
print("Selected image pool:", len(selected_image_ids))
print("Train images:", len(train_image_ids))
print("Val images:", len(val_image_ids))

In [ ]:
from pathlib import Path
import json

STAGE2_ROOT = Path("/home/CS159FinalProject/data/processed/stage2_instruction")

with open(STAGE2_ROOT / "shared_quality_pool.json", "r", encoding="utf-8") as f:
    pool_info = json.load(f)

with open(STAGE2_ROOT / "shared_split.json", "r", encoding="utf-8") as f:
    split_info = json.load(f)

selected_ids = set(pool_info["selected_image_ids"])
train_ids = set(split_info["train_image_ids"])
val_ids = set(split_info["val_image_ids"])

print("Using shared quality pool of size:", len(selected_ids))
print("Train images in split:", len(train_ids))
print("Val images in split:", len(val_ids))

for variant in ALL_VARIANTS:
    src = STAGE2_ROOT / variant / "stage2_dataset.jsonl"
    train_out = STAGE2_ROOT / variant / "stage2_train.jsonl"
    val_out = STAGE2_ROOT / variant / "stage2_val.jsonl"

    if not src.exists():
        raise FileNotFoundError(
            f"Missing dataset file for variant '{variant}': {src}"
        )

    rows = []
    with open(src, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            if row["image_id"] in selected_ids:
                rows.append(row)

    present_ids = {r["image_id"] for r in rows}
    missing_ids = selected_ids - present_ids
    if missing_ids:
        raise ValueError(
            f"Variant '{variant}' is missing {len(missing_ids)} selected image_ids. "
            f"First few missing: {sorted(missing_ids)[:10]}"
        )

    train_rows = [r for r in rows if r["image_id"] in train_ids]
    val_rows = [r for r in rows if r["image_id"] in val_ids]

    with open(train_out, "w", encoding="utf-8") as f:
        for row in train_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    with open(val_out, "w", encoding="utf-8") as f:
        for row in val_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"\nVariant: {variant}")
    print("  pooled rows:", len(rows))
    print("  pooled unique images:", len(present_ids))
    print("  train rows:", len(train_rows))
    print("  val rows:", len(val_rows))

In [ ]:
REASONING_WEIGHT = 1.5
DESCRIPTION_WEIGHT = 1.0
CONVERSATION_WEIGHT = 1.0

In [ ]:
import json
from pathlib import Path

import numpy as np
import jax.numpy as jnp
from PIL import Image

STAGE2_FEATURE_ROOT = Path("../data/processed/stage2_features")
STAGE2_FEATURE_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_ROOT_PATH = Path(IMAGE_ROOT)

# compile once
get_features_compiled = make_clip_feature_fn(clip_bundle)


def extract_stage2_features(variant, data_split="train"):
    variant_dir, _, tokenized_json = resolve_stage2_paths(STAGE2_ROOT, variant, data_split)

    if not tokenized_json.exists():
        raise FileNotFoundError(
            f"Stage-2 tokenized file not found yet: {tokenized_json}\n"
            "Run tokenize_stage2_variant(...) first."
        )

    with open(tokenized_json, "r", encoding="utf-8") as f:
        stage2_data = json.load(f)

    stats_path = variant_dir / f"stage2_feature_stats_{data_split}.json"

    created_features = 0
    skipped_features = 0
    missing_images = 0
    failed_images = 0

    feature_records = []

    for i, row in enumerate(stage2_data):
        image_name = row["image"]
        image_path = IMAGE_ROOT_PATH / image_name

        # save extracted image features
        feature_relpath = Path(image_name).with_suffix(".npy")
        feature_path = STAGE2_FEATURE_ROOT / feature_relpath
        feature_path.parent.mkdir(parents=True, exist_ok=True)

        record = {
            "image": image_name,
            "image_id": row.get("image_id"),
            "image_path": str(image_path),
            "feature_path": str(feature_path),
            "task_type": row.get("task_type"),
            "generator_model": row.get("generator_model"),
            "data_split": data_split,
            "variant": variant,
        }

        if feature_path.exists():
            skipped_features += 1
            record["status"] = "cached"
            feature_records.append(record)
            continue

        if not image_path.exists():
            missing_images += 1
            record["status"] = "missing_image"
            feature_records.append(record)
            print("Missing image:", image_path)
            continue

        try:
            img = Image.open(image_path).convert("RGB")
            clip_inputs = clip_bundle.processor(images=img, return_tensors="np")
            pixel_values = jnp.asarray(clip_inputs["pixel_values"])

            hidden_states_penultimate = get_features_compiled(pixel_values)

            vision_feats = np.array(
                jnp.asarray(hidden_states_penultimate[0], dtype=jnp.float32)
            )

            np.save(feature_path, vision_feats)

            created_features += 1
            record["status"] = "created"
            record["feature_shape"] = list(vision_feats.shape)

        except Exception as e:
            failed_images += 1
            record["status"] = "failed"
            record["error"] = str(e)
            print(f"Failed on image {image_name}: {e}")

        feature_records.append(record)

        if (i + 1) % 100 == 0 or (i + 1) == len(stage2_data):
            print(f"[{variant} | {data_split}] processed {i + 1}/{len(stage2_data)}")

    feature_stats = {
        "variant": variant,
        "data_split": data_split,
        "tokenized_json": str(tokenized_json),
        "feature_root": str(STAGE2_FEATURE_ROOT),
        "num_rows": len(stage2_data),
        "created_features": created_features,
        "skipped_features": skipped_features,
        "missing_images": missing_images,
        "failed_images": failed_images,
        "records": feature_records,
    }

    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(feature_stats, f, ensure_ascii=False, indent=2)

    print("Variant:", variant)
    print("Data split:", data_split)
    print("Created feature files:", created_features)
    print("Already existed:", skipped_features)
    print("Missing images:", missing_images)
    print("Failed images:", failed_images)
    print("Saved feature stats to:", stats_path)

    return {
        "variant": variant,
        "data_split": data_split,
        "stats_path": str(stats_path),
        "created_features": created_features,
        "skipped_features": skipped_features,
        "missing_images": missing_images,
        "failed_images": failed_images,
    }

In [ ]:
import json
from pathlib import Path


def build_stage2_manifest(variant, data_split="train"):
    variant_dir, _, tokenized_json = resolve_stage2_paths(STAGE2_ROOT, variant, data_split)

    if data_split == "train":
        manifest_json = variant_dir / "stage2_manifest_train.json"
    elif data_split == "val":
        manifest_json = variant_dir / "stage2_manifest_val.json"
    elif data_split == "full":
        manifest_json = variant_dir / "stage2_manifest_full.json"
    else:
        raise ValueError(f"data_split must be one of ['train', 'val', 'full'], got: {data_split}")

    if not tokenized_json.exists():
        raise FileNotFoundError(
            f"Stage-2 tokenized file not found yet: {tokenized_json}\n"
            "Run tokenize_stage2_variant(...) first."
        )

    with open(tokenized_json, "r", encoding="utf-8") as f:
        stage2_data = json.load(f)

    stage2_manifest = []
    missing_features = 0
    missing_feature_images = []

    for row in stage2_data:
        feature_relpath = Path(row["image"]).with_suffix(".npy")
        vision_path = STAGE2_FEATURE_ROOT / feature_relpath

        if vision_path.exists():
            stage2_manifest.append({
                "vision_path": str(vision_path),
                "input_ids": row["input_ids"],
                "labels": row["labels"],
                "image": row.get("image"),
                "image_id": row.get("image_id"),
                "task_type": row.get("task_type", "unknown"),
                "generator_model": row.get("generator_model"),
                "sample_id": row.get("sample_id"),
                "variant": variant,
                "data_split": data_split,
            })
        else:
            missing_features += 1
            missing_feature_images.append(row["image"])

    with open(manifest_json, "w", encoding="utf-8") as f:
        json.dump(stage2_manifest, f, ensure_ascii=False)

    print("Variant:", variant)
    print("Data split:", data_split)
    print("Stage-2 manifest size:", len(stage2_manifest))
    print("Missing feature files:", missing_features)
    if missing_feature_images:
        print("Images missing cached features:", missing_feature_images[:10])
    print("Saved to:", manifest_json)

    return {
        "variant": variant,
        "data_split": data_split,
        "manifest_json": str(manifest_json),
        "num_rows": len(stage2_manifest),
        "missing_features": missing_features,
    }

In [ ]:
prep_results = []

for variant in ALL_VARIANTS:
    for split in ["train", "val"]:
        print(f"\n===== PREPARING {variant} | {split} =====")
        tok_result = tokenize_stage2_variant(variant, split)
        feat_result = extract_stage2_features(variant, split)
        manifest_result = build_stage2_manifest(variant, split)

        prep_results.append({
            "variant": variant,
            "split": split,
            "tokenize": tok_result,
            "features": feat_result,
            "manifest": manifest_result,
        })

print("\nFinished preparing variants:", ALL_VARIANTS)

In [ ]:
# 3. Multimodal input builder

import jax.numpy as jnp

def make_multimodal_inputs(llama_model, projector, batch):
    vision_feats = jnp.asarray(batch["vision_feats"], dtype=jnp.float32)
    input_ids = jnp.asarray(batch["input_ids"], dtype=jnp.int32)
    labels = jnp.asarray(batch["labels"], dtype=jnp.int32)

    if vision_feats.ndim != 3:
        raise ValueError(
            f"vision_feats should have shape [batch, n_vis, vis_dim], got {vision_feats.shape}"
        )
    if input_ids.ndim != 2:
        raise ValueError(
            f"input_ids should have shape [batch, text_len], got {input_ids.shape}"
        )
    if labels.ndim != 2:
        raise ValueError(
            f"labels should have shape [batch, text_len], got {labels.shape}"
        )
    if input_ids.shape != labels.shape:
        raise ValueError(
            f"input_ids and labels must have the same shape, got {input_ids.shape} vs {labels.shape}"
        )

    bsz = vision_feats.shape[0]
    if input_ids.shape[0] != bsz:
        raise ValueError(
            f"Batch size mismatch: vision_feats batch={bsz}, input_ids batch={input_ids.shape[0]}"
        )

    # Project frozen visual features into the LLM embedding space
    vis_embeds = projector(vision_feats)

    # Text token embeddings
    txt_embeds = llama_model.embedder.encode(input_ids)

    # Keep dtypes aligned before concatenation
    txt_embeds = txt_embeds.astype(vis_embeds.dtype)

    if vis_embeds.ndim != 3 or txt_embeds.ndim != 3:
        raise ValueError(
            f"Expected 3D embeddings, got vis_embeds={vis_embeds.shape}, txt_embeds={txt_embeds.shape}"
        )

    if vis_embeds.shape[0] != txt_embeds.shape[0]:
        raise ValueError(
            f"Embedding batch mismatch: {vis_embeds.shape[0]} vs {txt_embeds.shape[0]}"
        )

    if vis_embeds.shape[-1] != txt_embeds.shape[-1]:
        raise ValueError(
            f"Embedding dim mismatch: {vis_embeds.shape[-1]} vs {txt_embeds.shape[-1]}"
        )

    # Image-first ordering: [visual tokens][text tokens]
    input_embeds = jnp.concatenate([vis_embeds, txt_embeds], axis=1)

    _, n_vis, _ = vis_embeds.shape
    _, tlen, _ = txt_embeds.shape
    seq_len = n_vis + tlen

    # Visual tokens are never prediction targets
    vis_labels = jnp.full((bsz, n_vis), -100, dtype=jnp.int32)
    full_labels = jnp.concatenate([vis_labels, labels], axis=1)

    # Standard absolute positions over the concatenated sequence
    positions = jnp.broadcast_to(
        jnp.arange(seq_len, dtype=jnp.int32)[None, :],
        (bsz, seq_len),
    )

    # Causal attention mask over the full multimodal sequence
    attention_mask = jnp.tril(
        jnp.ones((seq_len, seq_len), dtype=jnp.bool_)
    )
    attention_mask = jnp.broadcast_to(
        attention_mask[None, :, :],
        (bsz, seq_len, seq_len),
    )

    return {
        "input_embeds": input_embeds,
        "labels": full_labels,
        "positions": positions,
        "attention_mask": attention_mask,
        "n_vis_tokens": n_vis,
        "text_len": tlen,
        "seq_len": seq_len,
    }

In [ ]:
import optax
import pickle

STAGE2_LR = 2e-5
STAGE2_WEIGHT_DECAY = 0.0


def init_stage2_run():
    """
    Reset Stage 2 to a clean, comparable starting point.

    Returns:
        projector_state, llama_state, tx, opt_state
    """
    # rebuild a fresh projector skeleton
    projector = VisionProjector(
        in_dim=clip_bundle.hidden_size,
        out_dim=llama3_1b.config.embed_dim,
        rngs=nnx.Rngs(0),
    )

    # fresh base LLM state from the original loaded model
    # (this should be the same un-finetuned LLM every time)
    _, fresh_llama_state = nnx.split(llama3_1b)
    _, fresh_projector_state = nnx.split(projector)

    # restore projector from Stage 1 checkpoint
    with open(PROJECTOR_STATE_PATH, "rb") as f:
        fresh_projector_state = pickle.load(f)

    tx = optax.adamw(
        learning_rate=STAGE2_LR,
        weight_decay=STAGE2_WEIGHT_DECAY,
    )

    opt_state = tx.init({
        "projector": fresh_projector_state,
        "llama": fresh_llama_state,
    })

    print("Stage-2 optimizer initialized.")
    print("Learning rate:", STAGE2_LR)
    print("Weight decay:", STAGE2_WEIGHT_DECAY)

    return fresh_projector_state, fresh_llama_state, tx, opt_state

In [ ]:
def llama_forward_from_embeddings_stage2(
    llama_model,
    input_embeds,
    positions,
    attention_mask,
    cache=None,
):
    x = input_embeds
    running_cache = cache

    for layer in llama_model.layers:
        running_cache, x = layer(
            x=x,
            segment_pos=positions,
            cache=running_cache,
            attn_mask=attention_mask,
        )

    x = llama_model.final_norm(x)

    # Prefer tied embedding decode when available
    if hasattr(llama_model.embedder, "decode"):
        logits = llama_model.embedder.decode(x)
    elif hasattr(llama_model, "lm_head"):
        logits = llama_model.lm_head(x)
    else:
        raise AttributeError(
            "Neither llama_model.embedder.decode nor llama_model.lm_head is available."
        )

    return logits, running_cache

In [ ]:
@nnx.jit
def train_step_stage2(
    projector_state,
    llama_state,
    opt_state,
    batch,
):
    def loss_fn(projector_state, llama_state):
        projector = nnx.merge(projector_graphdef, projector_state)
        llama_model = nnx.merge(llama_graphdef, llama_state)

        mm = make_multimodal_inputs(llama_model, projector, batch)

        # IMPORTANT:
        # this cell assumes you already updated the helper cell so that
        # llama_forward_from_embeddings_stage2(...) exists.
        logits, _ = llama_forward_from_embeddings_stage2(
            llama_model,
            input_embeds=mm["input_embeds"],
            positions=mm["positions"],
            attention_mask=mm["attention_mask"],
            cache=None,
        )

        loss = masked_cross_entropy_loss(logits, mm["labels"])
        return loss

    loss, grads = jax.value_and_grad(loss_fn, argnums=(0, 1))(
        projector_state,
        llama_state,
    )
    projector_grads, llama_grads = grads

    updates, opt_state = tx.update(
        {
            "projector": projector_grads,
            "llama": llama_grads,
        },
        opt_state,
        {
            "projector": projector_state,
            "llama": llama_state,
        },
    )

    new_projector_state = optax.apply_updates(
        projector_state,
        updates["projector"],
    )
    new_llama_state = optax.apply_updates(
        llama_state,
        updates["llama"],
    )

    return new_projector_state, new_llama_state, opt_state, loss

In [ ]:
# Duplicate Stage-2 optimizer cell:
# do NOT reinitialize opt_state here.
# The real Stage-2 optimizer was already created in the earlier optimizer cell
# using the correct dict-shaped pytree:
# {
#   "projector": projector_state,
#   "llama": llama_state,
# }

print("Skipped duplicate Stage-2 optimizer initialization.")
print("Keep the earlier optimizer cell only.")
print("Expected optimizer pytree structure:")
print({"projector": "projector_state", "llama": "llama_state"})
print("Current learning rate:", STAGE2_LR)

In [ ]:
def stage2_collate_fn(batch):
    if len(batch) == 0:
        raise ValueError("stage2_collate_fn received an empty batch.")

    max_text_len = max(len(x["input_ids"]) for x in batch)

    vision_feats = []
    input_ids = []
    labels = []
    sample_weights = []

    expected_vis_shape = None

    for sample in batch:
        vf = np.load(sample["vision_path"], allow_pickle=False)

        if vf.dtype.kind == "V":
            vf = vf.view(ml_dtypes.bfloat16).astype(np.float32)
        else:
            vf = vf.astype(np.float32, copy=False)

        if vf.ndim != 2:
            raise ValueError(
                f"Expected vision feature shape [n_vis_tokens, vis_dim], got {vf.shape} "
                f"from {sample['vision_path']}"
            )

        if expected_vis_shape is None:
            expected_vis_shape = vf.shape
        elif vf.shape != expected_vis_shape:
            raise ValueError(
                f"Inconsistent vision feature shapes in batch: "
                f"expected {expected_vis_shape}, got {vf.shape} "
                f"from {sample['vision_path']}"
            )

        vision_feats.append(vf)
        input_ids.append(pad_list(sample["input_ids"], max_text_len, 0))
        labels.append(pad_list(sample["labels"], max_text_len, -100))

        task_type = sample.get("task_type", "unknown")
        if task_type == "complex_reasoning":
            sample_weights.append(REASONING_WEIGHT)
        elif task_type == "detailed_description":
            sample_weights.append(DESCRIPTION_WEIGHT)
        else:
            sample_weights.append(CONVERSATION_WEIGHT)

    return {
        "vision_feats": np.stack(vision_feats, axis=0).astype(np.float32, copy=False),
        "input_ids": np.asarray(input_ids, dtype=np.int32),
        "labels": np.asarray(labels, dtype=np.int32),
        "sample_weights": np.asarray(sample_weights, dtype=np.float32),
    }

In [ ]:
@nnx.jit
def train_step_stage2(
    projector_state,
    llama_state,
    opt_state,
    batch,
):
    def loss_fn(projector_state, llama_state):
        projector = nnx.merge(projector_graphdef, projector_state)
        llama_model = nnx.merge(llama_graphdef, llama_state)

        mm = make_multimodal_inputs(llama_model, projector, batch)

        logits, _ = llama_forward_from_embeddings_stage2(
            llama_model,
            input_embeds=mm["input_embeds"],
            positions=mm["positions"],
            attention_mask=mm["attention_mask"],
            cache=None,
        )

        vocab_size = logits.shape[-1]
        flat_logits = logits.reshape(-1, vocab_size)
        flat_labels = mm["labels"].reshape(-1)

        token_mask = (flat_labels != -100)
        safe_labels = jnp.where(token_mask, flat_labels, 0)

        log_probs = jax.nn.log_softmax(flat_logits, axis=-1)
        nll = -log_probs[jnp.arange(flat_labels.shape[0]), safe_labels]
        nll = nll.reshape(mm["labels"].shape)

        batch_size_local, seq_len = mm["labels"].shape

        sample_weights = jnp.asarray(batch["sample_weights"], dtype=jnp.float32)
        sample_weights = jnp.broadcast_to(
            sample_weights[:, None],
            (batch_size_local, seq_len),
        )

        token_mask_2d = token_mask.reshape(mm["labels"].shape).astype(jnp.float32)
        weighted_mask = token_mask_2d * sample_weights

        weighted_loss = (nll * weighted_mask).sum() / jnp.maximum(weighted_mask.sum(), 1.0)
        return weighted_loss

    loss, grads = jax.value_and_grad(loss_fn, argnums=(0, 1))(
        projector_state,
        llama_state,
    )
    projector_grads, llama_grads = grads

    updates, opt_state = tx.update(
        {
            "projector": projector_grads,
            "llama": llama_grads,
        },
        opt_state,
        {
            "projector": projector_state,
            "llama": llama_state,
        },
    )

    new_projector_state = optax.apply_updates(projector_state, updates["projector"])
    new_llama_state = optax.apply_updates(llama_state, updates["llama"])

    return new_projector_state, new_llama_state, opt_state, loss

In [ ]:
@nnx.jit
def eval_step_stage2(
    projector_state,
    llama_state,
    batch,
):
    projector = nnx.merge(projector_graphdef, projector_state)
    llama_model = nnx.merge(llama_graphdef, llama_state)

    mm = make_multimodal_inputs(llama_model, projector, batch)

    logits, _ = llama_forward_from_embeddings_stage2(
        llama_model,
        input_embeds=mm["input_embeds"],
        positions=mm["positions"],
        attention_mask=mm["attention_mask"],
        cache=None,
    )

    loss = masked_cross_entropy_loss(logits, mm["labels"])
    return loss


def evaluate_stage2(
    val_manifest_json,
    batch_size=8,
):
    global projector_state, llama_state

    with open(val_manifest_json, "r", encoding="utf-8") as f:
        manifest = json.load(f)

    losses = []

    for batch in iterate_stage2_minibatches(manifest, batch_size=batch_size, shuffle=False):
        loss = eval_step_stage2(
            projector_state,
            llama_state,
            batch,
        )
        losses.append(float(loss))

    mean_loss = sum(losses) / len(losses) if losses else None

    return {
        "mean_loss": mean_loss,
        "num_batches": len(losses),
        "batch_losses": losses,
    }

In [ ]:
# @nnx.jit
# def eval_step_stage2(
#     projector_state,
#     llama_state,
#     batch,
# ):
#     projector = nnx.merge(projector_graphdef, projector_state)
#     llama_model = nnx.merge(llama_graphdef, llama_state)

#     mm = make_multimodal_inputs(llama_model, projector, batch)

#     logits, _ = llama_forward_from_embeddings_stage2(
#         llama_model,
#         input_embeds=mm["input_embeds"],
#         positions=mm["positions"],
#         attention_mask=mm["attention_mask"],
#         cache=None,
#     )

#     loss = masked_cross_entropy_loss(logits, mm["labels"])
#     return loss

# import json

# def evaluate_stage2(
#     val_manifest_json,
#     batch_size=8,
# ):
#     with open(val_manifest_json, "r", encoding="utf-8") as f:
#         manifest = json.load(f)

#     losses = []

#     for batch in iterate_stage2_minibatches(manifest, batch_size=batch_size, shuffle=False):
#         loss = eval_step_stage2(
#             projector_state,
#             llama_state,
#             batch,
#         )
#         losses.append(float(loss))

#     mean_loss = sum(losses) / len(losses) if losses else None

#     return {
#         "mean_loss": mean_loss,
#         "num_batches": len(losses),
#         "batch_losses": losses,
#     }

In [ ]:
def run_stage2_training(
    manifest_json,
    val_manifest_json=None,
    num_epochs=1,
    batch_size=8,
    log_every_steps=20,
):
    global projector_state, llama_state, opt_state

    with open(manifest_json, "r", encoding="utf-8") as f:
        manifest = json.load(f)

    history = {
        "train_step_losses": [],
        "epoch_train_averages": [],
        "val_epoch_averages": [],
    }

    global_step = 0

    for epoch in range(num_epochs):
        epoch_losses = []

        print(f"\n===== Stage-2 Epoch {epoch + 1}/{num_epochs} =====")

        for batch in iterate_stage2_minibatches(manifest, batch_size=batch_size, shuffle=True):
            projector_state, llama_state, opt_state, loss = train_step_stage2(
                projector_state,
                llama_state,
                opt_state,
                batch,
            )

            loss_value = float(loss)
            history["train_step_losses"].append(loss_value)
            epoch_losses.append(loss_value)
            global_step += 1

            if log_every_steps and global_step % log_every_steps == 0:
                cumulative_mean = sum(history["train_step_losses"]) / len(history["train_step_losses"])
                print(
                    f"[train-step] epoch={epoch + 1} "
                    f"global_step={global_step} "
                    f"cumulative_mean_loss={cumulative_mean:.4f}"
                )

        epoch_train_avg = sum(epoch_losses) / len(epoch_losses) if epoch_losses else None
        history["epoch_train_averages"].append({
            "epoch": epoch + 1,
            "mean_loss": epoch_train_avg,
            "num_steps": len(epoch_losses),
        })

        if val_manifest_json is not None:
            val_result = evaluate_stage2(
                val_manifest_json=val_manifest_json,
                batch_size=batch_size,
            )
            history["val_epoch_averages"].append({
                "epoch": epoch + 1,
                "mean_loss": val_result["mean_loss"],
                "num_batches": val_result["num_batches"],
            })

            print(
                f"[val] epoch={epoch + 1} "
                f"mean_loss={val_result['mean_loss']:.4f}"
            )

    final_train_avg = sum(history["train_step_losses"]) / len(history["train_step_losses"])
    final_val_result = None
    if val_manifest_json is not None:
        final_val_result = evaluate_stage2(
            val_manifest_json=val_manifest_json,
            batch_size=batch_size,
        )

    print("\n===== Stage-2 Training Summary =====")
    print("Final overall train mean loss:", final_train_avg)
    print("Final overall val mean loss:  ", final_val_result["mean_loss"] if final_val_result else None)

    return {
        "history": history,
        "final_train_mean_loss": final_train_avg,
        "final_val_result": final_val_result,
    }

In [ ]:
import pickle
from pathlib import Path

RESET_ROOT = STAGE2_ROOT / "_fresh_state_snapshot"
RESET_ROOT.mkdir(parents=True, exist_ok=True)

PROJECTOR_RESET_PATH = RESET_ROOT / "projector_state.pkl"
LLAMA_RESET_PATH = RESET_ROOT / "llama_state.pkl"
OPT_RESET_PATH = RESET_ROOT / "opt_state.pkl"

def save_fresh_stage2_snapshot():
    with open(PROJECTOR_RESET_PATH, "wb") as f:
        pickle.dump(projector_state, f)
    with open(LLAMA_RESET_PATH, "wb") as f:
        pickle.dump(llama_state, f)
    with open(OPT_RESET_PATH, "wb") as f:
        pickle.dump(opt_state, f)

    print("Saved fresh stage-2 snapshot to:", RESET_ROOT)

def reset_stage2_states():
    global projector_state, llama_state, opt_state

    with open(PROJECTOR_RESET_PATH, "rb") as f:
        projector_state = pickle.load(f)
    with open(LLAMA_RESET_PATH, "rb") as f:
        llama_state = pickle.load(f)
    with open(OPT_RESET_PATH, "rb") as f:
        opt_state = pickle.load(f)

In [ ]:
save_fresh_stage2_snapshot()

In [ ]:
def run_stage2_experiment(
    variant,
    num_epochs=1,
    batch_size=8,
    log_every_steps=20,
):
    global projector_state, llama_state, tx, opt_state

    variant_root = STAGE2_ROOT / variant
    train_manifest = variant_root / "stage2_manifest_train.json"
    val_manifest = variant_root / "stage2_manifest_val.json"

    projector_state, llama_state, tx, opt_state = init_stage2_run()

    print(f"\n===== Starting baseline Stage-2 run for variant: {variant} =====")
    print("Train manifest:", train_manifest)
    print("Val manifest:", val_manifest)

    training_result = run_stage2_training(
        manifest_json=str(train_manifest),
        val_manifest_json=str(val_manifest),
        num_epochs=num_epochs,
        batch_size=batch_size,
        log_every_steps=log_every_steps,
    )

    return {
        "variant": variant,
        "train_result": training_result,
        "val_result": training_result["final_val_result"],
        "final_train_mean_loss": training_result["final_train_mean_loss"],
        "history": training_result["history"],
    }

In [ ]:
curriculum_weight_result = run_stage2_experiment(
    "llama",
    num_epochs=1,
    batch_size=8,
    log_every_steps=20,
)

In [ ]:
# from pathlib import Path
# import json
# import statistics
# from collections import Counter, defaultdict

# STAGE2_ROOT = Path("/home/CS159FinalProject/data/processed/stage2_instruction")

# QUALITY_STATS_JSON = STAGE2_ROOT / "dataset_quality_diagnostics.json"
# QUALITY_STATS_MD = STAGE2_ROOT / "dataset_quality_diagnostics.md"

# # use shared selected pool if available
# pool_path = STAGE2_ROOT / "shared_quality_pool.json"
# if pool_path.exists():
#     with open(pool_path, "r", encoding="utf-8") as f:
#         pool_info = json.load(f)
#     selected_ids = set(pool_info["selected_image_ids"])
#     print("Using shared selected pool:", len(selected_ids))
# else:
#     selected_ids = None
#     print("shared_quality_pool.json not found; auditing full datasets.")

# def safe_mean(xs):
#     return sum(xs) / len(xs) if xs else None

# def safe_median(xs):
#     return statistics.median(xs) if xs else None

# def word_count(text):
#     return len(str(text).split())

# def char_count(text):
#     return len(str(text))

# def load_rows(variant, selected_ids=None):
#     dataset_path = STAGE2_ROOT / variant / "stage2_dataset.jsonl"
#     if not dataset_path.exists():
#         raise FileNotFoundError(f"Missing dataset file: {dataset_path}")

#     rows = []
#     with open(dataset_path, "r", encoding="utf-8") as f:
#         for line in f:
#             line = line.strip()
#             if not line:
#                 continue
#             row = json.loads(line)
#             if selected_ids is not None and row["image_id"] not in selected_ids:
#                 continue
#             rows.append(row)
#     return rows

# diagnostics = []

# for variant in ALL_VARIANTS:
#     rows = load_rows(variant, selected_ids=selected_ids)

#     task_buckets = defaultdict(list)
#     for row in rows:
#         task_buckets[row["task_type"]].append(row)

#     response_counter = Counter(str(r.get("response", "")).strip() for r in rows)
#     duplicate_response_count = sum(1 for resp, c in response_counter.items() if resp and c > 1)

#     overall_prompt_words = [word_count(r.get("instruction", "")) for r in rows]
#     overall_response_words = [word_count(r.get("response", "")) for r in rows]
#     overall_response_chars = [char_count(r.get("response", "")) for r in rows]
#     empty_response_count = sum(1 for r in rows if not str(r.get("response", "")).strip())

#     variant_summary = {
#         "variant": variant,
#         "num_rows": len(rows),
#         "num_unique_images": len({r["image_id"] for r in rows}),
#         "num_unique_tasks": len({r["task_type"] for r in rows}),
#         "empty_response_count": empty_response_count,
#         "duplicate_response_count": duplicate_response_count,
#         "overall": {
#             "mean_instruction_words": safe_mean(overall_prompt_words),
#             "median_instruction_words": safe_median(overall_prompt_words),
#             "mean_response_words": safe_mean(overall_response_words),
#             "median_response_words": safe_median(overall_response_words),
#             "mean_response_chars": safe_mean(overall_response_chars),
#             "median_response_chars": safe_median(overall_response_chars),
#         },
#         "by_task": {},
#         "most_repeated_responses": [],
#     }

#     for task_type, task_rows in sorted(task_buckets.items()):
#         instr_words = [word_count(r.get("instruction", "")) for r in task_rows]
#         resp_words = [word_count(r.get("response", "")) for r in task_rows]
#         resp_chars = [char_count(r.get("response", "")) for r in task_rows]
#         empty_count = sum(1 for r in task_rows if not str(r.get("response", "")).strip())

#         task_resp_counter = Counter(str(r.get("response", "")).strip() for r in task_rows)
#         task_duplicate_response_count = sum(1 for resp, c in task_resp_counter.items() if resp and c > 1)

#         variant_summary["by_task"][task_type] = {
#             "num_rows": len(task_rows),
#             "num_unique_images": len({r["image_id"] for r in task_rows}),
#             "empty_response_count": empty_count,
#             "duplicate_response_count": task_duplicate_response_count,
#             "mean_instruction_words": safe_mean(instr_words),
#             "median_instruction_words": safe_median(instr_words),
#             "mean_response_words": safe_mean(resp_words),
#             "median_response_words": safe_median(resp_words),
#             "mean_response_chars": safe_mean(resp_chars),
#             "median_response_chars": safe_median(resp_chars),
#         }

#     repeated_examples = []
#     for resp, count in response_counter.most_common(10):
#         if resp and count > 1:
#             repeated_examples.append({
#                 "count": count,
#                 "response_preview": resp[:300],
#             })
#     variant_summary["most_repeated_responses"] = repeated_examples

#     diagnostics.append(variant_summary)

# with open(QUALITY_STATS_JSON, "w", encoding="utf-8") as f:
#     json.dump(
#         {
#             "all_variants": ALL_VARIANTS,
#             "selected_pool_size": len(selected_ids) if selected_ids is not None else None,
#             "diagnostics": diagnostics,
#         },
#         f,
#         ensure_ascii=False,
#         indent=2,
#     )

# with open(QUALITY_STATS_MD, "w", encoding="utf-8") as f:
#     f.write("# Dataset Quality Diagnostics\n\n")
#     f.write(f"Variants: {', '.join(ALL_VARIANTS)}\n\n")
#     if selected_ids is not None:
#         f.write(f"Selected shared pool size: {len(selected_ids)}\n\n")

#     for row in diagnostics:
#         f.write(f"## {row['variant']}\n\n")
#         f.write(f"- rows: {row['num_rows']}\n")
#         f.write(f"- unique images: {row['num_unique_images']}\n")
#         f.write(f"- unique tasks: {row['num_unique_tasks']}\n")
#         f.write(f"- empty responses: {row['empty_response_count']}\n")
#         f.write(f"- duplicate responses: {row['duplicate_response_count']}\n\n")

#         overall = row["overall"]
#         f.write("### Overall\n\n")
#         f.write(f"- mean instruction words: {overall['mean_instruction_words']}\n")
#         f.write(f"- median instruction words: {overall['median_instruction_words']}\n")
#         f.write(f"- mean response words: {overall['mean_response_words']}\n")
#         f.write(f"- median response words: {overall['median_response_words']}\n")
#         f.write(f"- mean response chars: {overall['mean_response_chars']}\n")
#         f.write(f"- median response chars: {overall['median_response_chars']}\n\n")

#         f.write("### By task\n\n")
#         for task_type, stats in row["by_task"].items():
#             f.write(f"#### {task_type}\n\n")
#             f.write(f"- rows: {stats['num_rows']}\n")
#             f.write(f"- unique images: {stats['num_unique_images']}\n")
#             f.write(f"- empty responses: {stats['empty_response_count']}\n")
#             f.write(f"- duplicate responses: {stats['duplicate_response_count']}\n")
#             f.write(f"- mean instruction words: {stats['mean_instruction_words']}\n")
#             f.write(f"- median instruction words: {stats['median_instruction_words']}\n")
#             f.write(f"- mean response words: {stats['mean_response_words']}\n")
#             f.write(f"- median response words: {stats['median_response_words']}\n")
#             f.write(f"- mean response chars: {stats['mean_response_chars']}\n")
#             f.write(f"- median response chars: {stats['median_response_chars']}\n\n")

#         if row["most_repeated_responses"]:
#             f.write("### Most repeated responses\n\n")
#             for item in row["most_repeated_responses"]:
#                 f.write(f"- count={item['count']}: {item['response_preview']}\n")
#             f.write("\n")

# print("Saved dataset quality diagnostics JSON to:", QUALITY_STATS_JSON)
# print("Saved dataset quality diagnostics markdown to:", QUALITY_STATS_MD)

In [ ]:
# import json
# from pathlib import Path

# STAGE2_ROOT = Path("/home/CS159FinalProject/data/processed/stage2_instruction")
# RESULTS_PATH = STAGE2_ROOT / "all_results_manual.json"

# # Set to None to automatically pick the next unfinished variant.
# # Or set to one of ALL_VARIANTS explicitly, e.g. "gemma", "qwen", "llama".
# CURRENT_VARIANT = None

# # Safety flag: keep False unless you intentionally want to rerun and replace a saved result.
# ALLOW_OVERWRITE = False

# if RESULTS_PATH.exists():
#     with open(RESULTS_PATH, "r", encoding="utf-8") as f:
#         all_results = json.load(f)
#     print("Loaded existing results from:", RESULTS_PATH)
# else:
#     all_results = {}
#     with open(RESULTS_PATH, "w", encoding="utf-8") as f:
#         json.dump(all_results, f, ensure_ascii=False, indent=2)
#     print("Created new results file at:", RESULTS_PATH)

# expected_variants = ALL_VARIANTS if "ALL_VARIANTS" in globals() else sorted(all_results.keys())
# missing_variants = [v for v in expected_variants if v not in all_results]

# print("Expected variants:", expected_variants)
# print("Already finished:", list(all_results.keys()))
# print("Still missing:", missing_variants)

# if CURRENT_VARIANT is None:
#     if not missing_variants:
#         print("All expected variants already have saved results.")
#     else:
#         CURRENT_VARIANT = missing_variants[0]
#         print("Auto-selected next unfinished variant:", CURRENT_VARIANT)
# else:
#     print("Manually selected variant:", CURRENT_VARIANT)

# if CURRENT_VARIANT is not None and CURRENT_VARIANT not in expected_variants:
#     raise ValueError(
#         f"CURRENT_VARIANT='{CURRENT_VARIANT}' is not in ALL_VARIANTS={expected_variants}"
#     )

# if (
#     CURRENT_VARIANT is not None
#     and CURRENT_VARIANT in all_results
#     and not ALLOW_OVERWRITE
# ):
#     raise ValueError(
#         f"Result for '{CURRENT_VARIANT}' already exists in {RESULTS_PATH}. "
#         "Choose another variant or set ALLOW_OVERWRITE=True to rerun it."
#     )

In [ ]:
# import json
# from pathlib import Path

# if CURRENT_VARIANT is None:
#     print("No variant selected to run.")
# else:
#     print(f"Running variant: {CURRENT_VARIANT}")

#     result = run_stage2_experiment(
#         CURRENT_VARIANT,
#         num_epochs=1,
#         batch_size=8,
#         log_every_steps=20,
#     )

#     all_results[CURRENT_VARIANT] = result

#     with open(RESULTS_PATH, "w", encoding="utf-8") as f:
#         json.dump(all_results, f, ensure_ascii=False, indent=2)

#     print(f"Saved result for {CURRENT_VARIANT} to:", RESULTS_PATH)
#     print("Currently stored variants:", list(all_results.keys()))

#     remaining_after_run = [v for v in expected_variants if v not in all_results]
#     print("Remaining variants after this run:", remaining_after_run)

In [ ]:
# from pathlib import Path
# import json
# from collections import Counter

# STAGE2_ROOT = Path("/home/CS159FinalProject/data/processed/stage2_instruction")

# # Compare prompts on the shared selected image pool if available
# pool_path = STAGE2_ROOT / "shared_quality_pool.json"
# if pool_path.exists():
#     with open(pool_path, "r", encoding="utf-8") as f:
#         pool_info = json.load(f)
#     selected_ids = set(pool_info["selected_image_ids"])
#     print("Using selected shared image pool:", len(selected_ids))
# else:
#     selected_ids = None
#     print("shared_quality_pool.json not found, auditing full datasets instead.")

# PROMPT_REFERENCE_VARIANT = ALL_VARIANTS[0]
# print("Prompt reference variant:", PROMPT_REFERENCE_VARIANT)

# def load_variant_prompt_map(variant, selected_ids=None):
#     dataset_path = STAGE2_ROOT / variant / "stage2_dataset.jsonl"
#     if not dataset_path.exists():
#         raise FileNotFoundError(f"Missing dataset file: {dataset_path}")

#     prompt_map = {}
#     duplicate_keys = []
#     task_counter = Counter()

#     with open(dataset_path, "r", encoding="utf-8") as f:
#         for line in f:
#             line = line.strip()
#             if not line:
#                 continue
#             row = json.loads(line)

#             image_id = row["image_id"]
#             if selected_ids is not None and image_id not in selected_ids:
#                 continue

#             task_type = row["task_type"]
#             instruction = row["instruction"]
#             key = (image_id, task_type)

#             if key in prompt_map:
#                 duplicate_keys.append(key)
#             prompt_map[key] = instruction
#             task_counter[task_type] += 1

#     return prompt_map, duplicate_keys, task_counter

# reference_map, ref_duplicates, ref_task_counts = load_variant_prompt_map(
#     PROMPT_REFERENCE_VARIANT,
#     selected_ids=selected_ids,
# )

# print("\n=== Reference variant summary ===")
# print("Reference keys:", len(reference_map))
# print("Reference task counts:", dict(ref_task_counts))
# print("Reference duplicate keys:", len(ref_duplicates))

# audit_rows = []
# mismatch_examples = []

# for variant in ALL_VARIANTS:
#     prompt_map, duplicates, task_counts = load_variant_prompt_map(
#         variant,
#         selected_ids=selected_ids,
#     )

#     ref_keys = set(reference_map.keys())
#     variant_keys = set(prompt_map.keys())

#     missing_vs_ref = sorted(ref_keys - variant_keys)
#     extra_vs_ref = sorted(variant_keys - ref_keys)

#     instruction_mismatch_count = 0
#     shared_keys = sorted(ref_keys & variant_keys)

#     for key in shared_keys:
#         ref_instruction = reference_map[key]
#         variant_instruction = prompt_map[key]

#         if ref_instruction != variant_instruction:
#             instruction_mismatch_count += 1
#             if len(mismatch_examples) < 10:
#                 mismatch_examples.append({
#                     "variant": variant,
#                     "image_id": key[0],
#                     "task_type": key[1],
#                     "reference_instruction": ref_instruction,
#                     "variant_instruction": variant_instruction,
#                 })

#     row = {
#         "variant": variant,
#         "num_keys": len(prompt_map),
#         "task_counts": dict(task_counts),
#         "duplicate_key_count": len(duplicates),
#         "missing_vs_reference_count": len(missing_vs_ref),
#         "extra_vs_reference_count": len(extra_vs_ref),
#         "instruction_mismatch_count": instruction_mismatch_count,
#         "missing_vs_reference_examples": missing_vs_ref[:10],
#         "extra_vs_reference_examples": extra_vs_ref[:10],
#     }
#     audit_rows.append(row)

# print("\n=== Prompt alignment audit ===")
# for row in audit_rows:
#     print(
#         f"{row['variant']}: "
#         f"keys={row['num_keys']} | "
#         f"duplicates={row['duplicate_key_count']} | "
#         f"missing_vs_ref={row['missing_vs_reference_count']} | "
#         f"extra_vs_ref={row['extra_vs_reference_count']} | "
#         f"instruction_mismatches={row['instruction_mismatch_count']}"
#     )

# if mismatch_examples:
#     print("\n=== First few instruction mismatch examples ===")
#     for ex in mismatch_examples:
#         print(f"\nVariant: {ex['variant']}")
#         print(f"image_id: {ex['image_id']}")
#         print(f"task_type: {ex['task_type']}")
#         print("REFERENCE instruction:")
#         print(ex["reference_instruction"])
#         print("VARIANT instruction:")
#         print(ex["variant_instruction"])
# else:
#     print("\nNo instruction mismatches found against the reference variant.")

# audit_summary = {
#     "reference_variant": PROMPT_REFERENCE_VARIANT,
#     "selected_pool_size": len(selected_ids) if selected_ids is not None else None,
#     "audit_rows": audit_rows,
#     "mismatch_examples": mismatch_examples,
# }

# audit_path = STAGE2_ROOT / "prompt_alignment_audit.json"
# with open(audit_path, "w", encoding="utf-8") as f:
#     json.dump(audit_summary, f, ensure_ascii=False, indent=2)

# print("\nSaved prompt alignment audit to:", audit_path)

In [ ]:
# from pathlib import Path
# import json

# STAGE2_ROOT = Path("/home/CS159FinalProject/data/processed/stage2_instruction")
# RESULTS_PATH = STAGE2_ROOT / "all_results_manual.json"
# ENGINE_SUMMARY_PATH = STAGE2_ROOT / "engine_comparison_summary.json"

# EXPECTED_VARIANTS = ["gemma", "qwen", "llama"]

# if not RESULTS_PATH.exists():
#     raise FileNotFoundError(f"Missing results file: {RESULTS_PATH}")

# with open(RESULTS_PATH, "r", encoding="utf-8") as f:
#     all_results = json.load(f)

# missing_variants = [v for v in EXPECTED_VARIANTS if v not in all_results]
# if missing_variants:
#     raise ValueError(f"Missing variants in results file: {missing_variants}")

# def first_existing(d, keys):
#     if not isinstance(d, dict):
#         return None, None
#     for k in keys:
#         v = d.get(k)
#         if v is not None:
#             return v, k
#     return None, None

# rows = []
# for variant in EXPECTED_VARIANTS:
#     result = all_results[variant]

#     train_value = result.get("final_train_mean_loss")
#     train_key = "final_train_mean_loss" if train_value is not None else None

#     val_result = result.get("val_result", {})
#     val_value, val_key = first_existing(
#         val_result,
#         [
#             "final_val_mean_loss",
#             "final_overall_val_mean_loss",
#             "overall_val_mean_loss",
#             "val_mean_loss",
#             "mean_loss",
#         ],
#     )

#     print(f"\nVariant: {variant}")
#     print("Top-level keys:", sorted(result.keys()))
#     print("val_result keys:", sorted(val_result.keys()) if isinstance(val_result, dict) else val_result)
#     print("Train metric used:", train_key, "=", train_value)
#     print("Val metric used:", val_key, "=", val_value)

#     if val_value is None:
#         raise ValueError(
#             f"Could not find a validation-loss key inside val_result for variant '{variant}'. "
#             f"See printed val_result keys above."
#         )

#     rows.append({
#         "variant": variant,
#         "final_train_mean_loss": train_value,
#         "final_val_mean_loss": val_value,
#         "train_key_used": train_key,
#         "val_key_used": f"val_result.{val_key}",
#     })

# rows = sorted(rows, key=lambda x: x["final_val_mean_loss"])
# best_variant = rows[0]["variant"]

# summary = {
#     "ranking_metric": "final_val_mean_loss_lower_is_better",
#     "best_variant": best_variant,
#     "variant_summaries": rows,
# }

# with open(ENGINE_SUMMARY_PATH, "w", encoding="utf-8") as f:
#     json.dump(summary, f, ensure_ascii=False, indent=2)

# print("\nSaved engine comparison summary to:", ENGINE_SUMMARY_PATH)
# print("Best variant:", best_variant)
# print("\nRanking by final_val_mean_loss:")
# for i, row in enumerate(rows, start=1):
#     train_str = f"{row['final_train_mean_loss']:.4f}" if row["final_train_mean_loss"] is not None else "None"
#     val_str = f"{row['final_val_mean_loss']:.4f}"
#     print(f"{i}. {row['variant']} | train={train_str} | val={val_str}")

In [ ]:
# from pathlib import Path
# import json

# STAGE2_ROOT = Path("/home/CS159FinalProject/data/processed/stage2_instruction")

# ENGINE_SUMMARY_PATH = STAGE2_ROOT / "engine_comparison_summary.json"
# POOL_INFO_PATH = STAGE2_ROOT / "shared_quality_pool.json"
# SPLIT_INFO_PATH = STAGE2_ROOT / "shared_split.json"

# if not ENGINE_SUMMARY_PATH.exists():
#     raise FileNotFoundError(
#         f"Missing engine comparison summary: {ENGINE_SUMMARY_PATH}\n"
#         "Run the quality comparison cells first so the notebook knows which variant performed best."
#     )

# with open(ENGINE_SUMMARY_PATH, "r", encoding="utf-8") as f:
#     engine_summary = json.load(f)

# ranking = engine_summary.get("variant_summaries", [])
# if not ranking:
#     raise ValueError(
#         "engine_comparison_summary.json has no variant_summaries entries. "
#         "Finish the quality experiment before running quantity ablation."
#     )

# QUANTITY_SOURCE_VARIANT = engine_summary.get("best_variant") or ranking[0]["variant"]

# # Base quantity should match the quality-comparison pool size (your intended 5000-image experiment).
# if "QUALITY_IMAGE_COUNT" in globals():
#     QUANTITY_BASE_TOTAL_IMAGES = int(QUALITY_IMAGE_COUNT)
# elif POOL_INFO_PATH.exists():
#     with open(POOL_INFO_PATH, "r", encoding="utf-8") as f:
#         pool_info = json.load(f)
#     QUANTITY_BASE_TOTAL_IMAGES = int(pool_info.get("quality_image_count", 5000))
# else:
#     QUANTITY_BASE_TOTAL_IMAGES = 5000

# # Keep validation size consistent with your earlier split if available.
# if "VAL_IMAGE_COUNT" in globals():
#     QUANTITY_VAL_IMAGE_COUNT = int(VAL_IMAGE_COUNT)
# elif SPLIT_INFO_PATH.exists():
#     with open(SPLIT_INFO_PATH, "r", encoding="utf-8") as f:
#         split_info = json.load(f)
#     QUANTITY_VAL_IMAGE_COUNT = int(split_info.get("num_val_images", 1000))
# else:
#     QUANTITY_VAL_IMAGE_COUNT = 1000

# # Your stated plan: best dataset, then +/- 1000 images around it.
# QUANTITY_LEVELS = [
#     QUANTITY_BASE_TOTAL_IMAGES -2000,
#     QUANTITY_BASE_TOTAL_IMAGES -1000,
#     QUANTITY_BASE_TOTAL_IMAGES,
# ]

# if min(QUANTITY_LEVELS) <= 0:
#     raise ValueError(f"Invalid quantity levels: {QUANTITY_LEVELS}")

# QUANTITY_SPLIT_SEED = 123

# print("Best quality variant selected for quantity ablation:", QUANTITY_SOURCE_VARIANT)
# print("Base total image count:", QUANTITY_BASE_TOTAL_IMAGES)
# print("Validation image count target:", QUANTITY_VAL_IMAGE_COUNT)
# print("Quantity levels (total images):", QUANTITY_LEVELS)

In [ ]:
# from pathlib import Path
# import json
# import random

# STAGE2_ROOT = Path("/home/CS159FinalProject/data/processed/stage2_instruction")

# with open(STAGE2_ROOT / "shared_quality_pool.json", "r", encoding="utf-8") as f:
#     pool_info = json.load(f)

# with open(STAGE2_ROOT / "shared_split.json", "r", encoding="utf-8") as f:
#     split_info = json.load(f)

# selected_ids = set(pool_info["selected_image_ids"])
# base_train_ids = sorted(split_info["train_image_ids"])
# base_val_ids = sorted(split_info["val_image_ids"])

# num_base_train = len(base_train_ids)
# num_base_val = len(base_val_ids)

# print("Quantity source variant:", QUANTITY_SOURCE_VARIANT)
# print("Available pooled train images:", num_base_train)
# print("Validation images stay fixed:", num_base_val)

# source_dataset = STAGE2_ROOT / QUANTITY_SOURCE_VARIANT / "stage2_dataset.jsonl"
# if not source_dataset.exists():
#     raise FileNotFoundError(f"Missing source dataset: {source_dataset}")

# rng = random.Random(QUANTITY_SPLIT_SEED)
# shuffled_train_ids = base_train_ids[:]
# rng.shuffle(shuffled_train_ids)

# quantity_root = STAGE2_ROOT / "quantity_ablation"
# quantity_root.mkdir(parents=True, exist_ok=True)

# quantity_variants = []

# for qty in QUANTITY_LEVELS:
#     # qty is TOTAL images for this experiment, not just train images
#     qty_train_count = qty - num_base_val

#     if qty_train_count <= 0:
#         raise ValueError(
#             f"Requested total quantity {qty}, but fixed validation size is {num_base_val}, "
#             f"so train size would be non-positive."
#         )

#     if qty_train_count > num_base_train:
#         raise ValueError(
#             f"Requested total quantity {qty} -> needs {qty_train_count} train images, "
#             f"but only {num_base_train} pooled train images are available."
#         )

#     qty_variant = f"{QUANTITY_SOURCE_VARIANT}_qty_{qty}"
#     qty_dir = quantity_root / qty_variant
#     qty_dir.mkdir(parents=True, exist_ok=True)

#     qty_train_ids = set(shuffled_train_ids[:qty_train_count])
#     qty_val_ids = set(base_val_ids)

#     qty_rows = []
#     with open(source_dataset, "r", encoding="utf-8") as f:
#         for line in f:
#             line = line.strip()
#             if not line:
#                 continue
#             row = json.loads(line)
#             if row["image_id"] in qty_train_ids or row["image_id"] in qty_val_ids:
#                 qty_rows.append(row)

#     train_rows = [r for r in qty_rows if r["image_id"] in qty_train_ids]
#     val_rows = [r for r in qty_rows if r["image_id"] in qty_val_ids]

#     with open(qty_dir / "stage2_train.jsonl", "w", encoding="utf-8") as f:
#         for row in train_rows:
#             f.write(json.dumps(row, ensure_ascii=False) + "\n")

#     with open(qty_dir / "stage2_val.jsonl", "w", encoding="utf-8") as f:
#         for row in val_rows:
#             f.write(json.dumps(row, ensure_ascii=False) + "\n")

#     metadata = {
#         "source_variant": QUANTITY_SOURCE_VARIANT,
#         "quantity_level_total_images": qty,
#         "num_train_images": len(qty_train_ids),
#         "num_val_images": len(qty_val_ids),
#         "num_total_images": len(qty_train_ids) + len(qty_val_ids),
#         "train_image_ids": sorted(qty_train_ids),
#         "val_image_ids": sorted(qty_val_ids),
#         "split_seed": QUANTITY_SPLIT_SEED,
#     }

#     with open(qty_dir / "metadata.json", "w", encoding="utf-8") as f:
#         json.dump(metadata, f, ensure_ascii=False, indent=2)

#     quantity_variants.append(qty_variant)

#     print(
#         f"Built {qty_variant}: "
#         f"train_images={len(qty_train_ids)}, "
#         f"val_images={len(qty_val_ids)}, "
#         f"total_images={len(qty_train_ids) + len(qty_val_ids)}"
#     )

# print("\nQuantity ablation variants prepared:")
# print(quantity_variants)

In [ ]:
# from pathlib import Path
# import shutil
# import json

# STAGE2_ROOT = Path("/home/CS159FinalProject/data/processed/stage2_instruction")

# QUANTITY_REGISTRATION_PATH = STAGE2_ROOT / "quantity_registration_status.json"

# required_files = ["stage2_train.jsonl", "stage2_val.jsonl", "metadata.json"]
# quantity_variants = [f"{QUANTITY_SOURCE_VARIANT}_qty_{q}" for q in QUANTITY_LEVELS]

# try:
#     with open(QUANTITY_REGISTRATION_PATH, "r", encoding="utf-8") as f:
#         registration_status = json.load(f)
#     print("Loaded existing quantity registration status.")
# except FileNotFoundError:
#     registration_status = {}
#     print("Starting new quantity registration status store.")

# for qty_variant in quantity_variants:
#     src_dir = STAGE2_ROOT / "quantity_ablation" / qty_variant
#     dst_dir = STAGE2_ROOT / qty_variant

#     if not src_dir.exists():
#         raise FileNotFoundError(f"Missing quantity source directory: {src_dir}")

#     missing_files = [name for name in required_files if not (src_dir / name).exists()]
#     if missing_files:
#         raise FileNotFoundError(
#             f"Cannot register {qty_variant}. Missing files in {src_dir}: {missing_files}"
#         )

#     dst_dir.mkdir(parents=True, exist_ok=True)

#     copied_files = []
#     for filename in required_files:
#         src_file = src_dir / filename
#         dst_file = dst_dir / filename
#         shutil.copy2(src_file, dst_file)
#         copied_files.append(str(dst_file))

#     registration_status[qty_variant] = {
#         "status": "registered",
#         "source_dir": str(src_dir),
#         "dest_dir": str(dst_dir),
#         "files": copied_files,
#     }

#     with open(QUANTITY_REGISTRATION_PATH, "w", encoding="utf-8") as f:
#         json.dump(registration_status, f, ensure_ascii=False, indent=2)

#     print(f"Registered quantity variant at: {dst_dir}")

# print("\nQuantity variants ready in normal STAGE2_ROOT layout:")
# print(quantity_variants)
# print("Saved registration status to:", QUANTITY_REGISTRATION_PATH)

In [ ]:
# import json
# from pathlib import Path

# QUANTITY_PREP_STATUS_PATH = STAGE2_ROOT / "quantity_prep_status.json"

# try:
#     with open(QUANTITY_PREP_STATUS_PATH, "r", encoding="utf-8") as f:
#         quantity_prep_status = json.load(f)
#     print("Loaded existing quantity prep status.")
# except FileNotFoundError:
#     quantity_prep_status = {}
#     print("Starting new quantity prep status store.")

# quantity_variants = [f"{QUANTITY_SOURCE_VARIANT}_qty_{q}" for q in QUANTITY_LEVELS]

# for variant in quantity_variants:
#     variant_root = STAGE2_ROOT / variant

#     train_manifest = variant_root / "stage2_manifest_train.json"
#     val_manifest = variant_root / "stage2_manifest_val.json"

#     already_ready = train_manifest.exists() and val_manifest.exists()

#     if already_ready:
#         print(f"\nSkipping already prepared quantity variant: {variant}")
#         quantity_prep_status[variant] = {
#             "status": "ready",
#             "train_manifest": str(train_manifest),
#             "val_manifest": str(val_manifest),
#         }
#         with open(QUANTITY_PREP_STATUS_PATH, "w", encoding="utf-8") as f:
#             json.dump(quantity_prep_status, f, ensure_ascii=False, indent=2)
#         continue

#     variant_log = {"status": "started", "splits": {}}
#     quantity_prep_status[variant] = variant_log

#     print(f"\n================ PREPARING QUANTITY VARIANT: {variant} ================\n")

#     for split in ["train", "val"]:
#         print(f"----- {variant} | {split} -----")

#         tok_result = tokenize_stage2_variant(variant, split)
#         feat_result = extract_stage2_features(variant, split)
#         manifest_result = build_stage2_manifest(variant, split)

#         variant_log["splits"][split] = {
#             "tokenize": tok_result,
#             "features": feat_result,
#             "manifest": manifest_result,
#         }

#         quantity_prep_status[variant] = variant_log
#         with open(QUANTITY_PREP_STATUS_PATH, "w", encoding="utf-8") as f:
#             json.dump(quantity_prep_status, f, ensure_ascii=False, indent=2)

#     variant_log["status"] = "ready"
#     variant_log["train_manifest"] = str(train_manifest)
#     variant_log["val_manifest"] = str(val_manifest)
#     quantity_prep_status[variant] = variant_log

#     with open(QUANTITY_PREP_STATUS_PATH, "w", encoding="utf-8") as f:
#         json.dump(quantity_prep_status, f, ensure_ascii=False, indent=2)

#     print(f"Finished preparing {variant}")

# print("\nPrepared quantity variants:")
# for variant in quantity_variants:
#     status = quantity_prep_status.get(variant, {}).get("status", "missing")
#     print(f"- {variant}: {status}")

# print("\nSaved prep status to:", QUANTITY_PREP_STATUS_PATH)

In [ ]:
# import json
# from pathlib import Path

# QUANTITY_RESULTS_PATH = STAGE2_ROOT / "quantity_results.json"

# # Safety flag: keep False unless you intentionally want to rerun saved quantity experiments.
# ALLOW_QUANTITY_OVERWRITE = False

# try:
#     with open(QUANTITY_RESULTS_PATH, "r", encoding="utf-8") as f:
#         quantity_results = json.load(f)
#     print("Loaded existing quantity results:", list(quantity_results.keys()))
# except FileNotFoundError:
#     quantity_results = {}
#     print("Starting new quantity results store.")

# quantity_variants = [f"{QUANTITY_SOURCE_VARIANT}_qty_{q}" for q in QUANTITY_LEVELS]

# print("\nPlanned quantity variants:", quantity_variants)

# for variant in quantity_variants:
#     if variant in quantity_results and not ALLOW_QUANTITY_OVERWRITE:
#         print(f"\nSkipping already finished quantity variant: {variant}")
#         continue

#     print(f"\n================ RUNNING QUANTITY EXPERIMENT: {variant} ================\n")

#     result = run_stage2_experiment(
#         variant,
#         num_epochs=1,
#         batch_size=8,
#         log_every_steps=20,
#     )

#     quantity_results[variant] = result

#     with open(QUANTITY_RESULTS_PATH, "w", encoding="utf-8") as f:
#         json.dump(quantity_results, f, ensure_ascii=False, indent=2)

#     print(f"Saved quantity result for {variant} to:", QUANTITY_RESULTS_PATH)

# print("\nFinished quantity experiments for:", list(quantity_results.keys()))
# print("Results file:", QUANTITY_RESULTS_PATH)

In [ ]:
# import json
# from pathlib import Path

# QUANTITY_RESULTS_PATH = STAGE2_ROOT / "quantity_results.json"

# if "quantity_results" not in globals() or not quantity_results:
#     if not QUANTITY_RESULTS_PATH.exists():
#         raise FileNotFoundError(
#             f"Missing quantity results file: {QUANTITY_RESULTS_PATH}\n"
#             "Run the quantity experiment cell first."
#         )
#     with open(QUANTITY_RESULTS_PATH, "r", encoding="utf-8") as f:
#         quantity_results = json.load(f)
#     print("Loaded quantity results from:", QUANTITY_RESULTS_PATH)
# else:
#     print("Using in-memory quantity_results.")

# quantity_ranking = []

# for variant, result in quantity_results.items():
#     train_loss = result.get("final_train_mean_loss", None)
#     val_result = result.get("val_result") or {}
#     val_loss = val_result.get("mean_loss", None)
#     qty = int(variant.split("_qty_")[-1])

#     quantity_ranking.append({
#         "qty": qty,
#         "variant": variant,
#         "train_loss": train_loss,
#         "val_loss": val_loss,
#     })

# quantity_ranking = sorted(quantity_ranking, key=lambda x: x["qty"])

# print("=== Quantity ablation summary ===")
# for row in quantity_ranking:
#     train_str = f"{row['train_loss']:.4f}" if row["train_loss"] is not None else "missing"
#     val_str = f"{row['val_loss']:.4f}" if row["val_loss"] is not None else "missing"
#     print(
#         f"{row['variant']}: "
#         f"quantity={row['qty']} | "
#         f"final_train_mean_loss={train_str} | "
#         f"final_val_mean_loss={val_str}"
#     )

# print("\n=== Cumulative train averages by quantity ===")
# for row in quantity_ranking:
#     print(f"\n--- {row['variant']} ---")
#     logs = quantity_results[row["variant"]].get("history", {}).get("train_cumulative_averages", [])
#     if not logs:
#         print("No cumulative logs found.")
#         continue

#     for log_row in logs:
#         mean_loss = log_row.get("mean_loss", None)
#         mean_str = f"{mean_loss:.4f}" if mean_loss is not None else "missing"
#         print(
#             f"epoch={log_row.get('epoch', 'NA')} "
#             f"step={log_row.get('global_step', 'NA')} "
#             f"cumulative_mean_loss={mean_str}"
#         )

# print("\n=== Quantity trend by final val loss ===")
# for row in quantity_ranking:
#     val_str = f"{row['val_loss']:.4f}" if row["val_loss"] is not None else "missing"
#     print(f"{row['qty']} images -> {val_str}")

# best_qty_row = min(
#     quantity_ranking,
#     key=lambda x: x["val_loss"] if x["val_loss"] is not None else float("inf")
# ) if quantity_ranking else None

# if best_qty_row is not None:
#     print(
#         f"\nBest quantity by final val loss: "
#         f"{best_qty_row['qty']} images ({best_qty_row['variant']})"
#     )

In [ ]:
# from pathlib import Path
# import json
# import random
# from collections import defaultdict

# STAGE2_ROOT = Path("/home/CS159FinalProject/data/processed/stage2_instruction")

# QUAL_SAMPLE_SEED = 42
# QUAL_SAMPLES_PER_TASK = 4
# QUAL_OUTPUT_JSON = STAGE2_ROOT / "qualitative_comparison_samples.json"
# QUAL_OUTPUT_MD = STAGE2_ROOT / "qualitative_comparison_samples.md"

# # load shared selected pool if available
# pool_path = STAGE2_ROOT / "shared_quality_pool.json"
# if pool_path.exists():
#     with open(pool_path, "r", encoding="utf-8") as f:
#         pool_info = json.load(f)
#     selected_ids = set(pool_info["selected_image_ids"])
#     print("Using shared selected pool:", len(selected_ids))
# else:
#     selected_ids = None
#     print("shared_quality_pool.json not found; using full datasets.")

# def load_rows_by_key(variant, selected_ids=None):
#     dataset_path = STAGE2_ROOT / variant / "stage2_dataset.jsonl"
#     if not dataset_path.exists():
#         raise FileNotFoundError(f"Missing dataset file: {dataset_path}")

#     rows_by_key = {}
#     with open(dataset_path, "r", encoding="utf-8") as f:
#         for line in f:
#             line = line.strip()
#             if not line:
#                 continue
#             row = json.loads(line)

#             if selected_ids is not None and row["image_id"] not in selected_ids:
#                 continue

#             key = (row["image_id"], row["task_type"])
#             rows_by_key[key] = row

#     return rows_by_key

# variant_maps = {variant: load_rows_by_key(variant, selected_ids=selected_ids) for variant in ALL_VARIANTS}

# common_keys = None
# for variant in ALL_VARIANTS:
#     keys = set(variant_maps[variant].keys())
#     common_keys = keys if common_keys is None else (common_keys & keys)

# common_keys = sorted(common_keys) if common_keys is not None else []
# print("Common keys across all variants:", len(common_keys))

# # keep only keys whose instruction text is identical across all variants
# aligned_keys = []
# for key in common_keys:
#     instructions = [variant_maps[v][key]["instruction"] for v in ALL_VARIANTS]
#     if len(set(instructions)) == 1:
#         aligned_keys.append(key)

# print("Instruction-aligned common keys:", len(aligned_keys))

# # group by task type
# keys_by_task = defaultdict(list)
# for key in aligned_keys:
#     image_id, task_type = key
#     keys_by_task[task_type].append(key)

# rng = random.Random(QUAL_SAMPLE_SEED)

# selected_keys = []
# for task_type, keys in sorted(keys_by_task.items()):
#     keys_copy = keys[:]
#     rng.shuffle(keys_copy)
#     selected_keys.extend(keys_copy[:QUAL_SAMPLES_PER_TASK])

# print("Selected qualitative examples:", len(selected_keys))

# samples = []
# for key in selected_keys:
#     image_id, task_type = key
#     base_row = variant_maps[ALL_VARIANTS[0]][key]

#     sample = {
#         "image_id": image_id,
#         "task_type": task_type,
#         "instruction": base_row["instruction"],
#         "responses": {},
#     }

#     for variant in ALL_VARIANTS:
#         sample["responses"][variant] = variant_maps[variant][key]["response"]

#     samples.append(sample)

# with open(QUAL_OUTPUT_JSON, "w", encoding="utf-8") as f:
#     json.dump(
#         {
#             "all_variants": ALL_VARIANTS,
#             "num_samples": len(samples),
#             "samples": samples,
#         },
#         f,
#         ensure_ascii=False,
#         indent=2,
#     )

# with open(QUAL_OUTPUT_MD, "w", encoding="utf-8") as f:
#     f.write("# Qualitative Comparison Samples\n\n")
#     f.write(f"Variants: {', '.join(ALL_VARIANTS)}\n\n")
#     f.write(f"Number of samples: {len(samples)}\n\n")

#     for i, sample in enumerate(samples, start=1):
#         f.write(f"## Sample {i}\n\n")
#         f.write(f"- **image_id:** {sample['image_id']}\n")
#         f.write(f"- **task_type:** {sample['task_type']}\n\n")
#         f.write("**Instruction**\n\n")
#         f.write(sample["instruction"] + "\n\n")

#         for variant in ALL_VARIANTS:
#             f.write(f"### {variant}\n\n")
#             f.write(sample["responses"][variant] + "\n\n")

# print("Saved qualitative JSON to:", QUAL_OUTPUT_JSON)
# print("Saved qualitative markdown to:", QUAL_OUTPUT_MD)

In [ ]:
import sys
print(sys.executable)
import sys
!{sys.executable} -m pip install matplotlib

In [ ]:
# from pathlib import Path
# import json
# import math
# import matplotlib.pyplot as plt

# STAGE2_ROOT = Path("/home/CS159FinalProject/data/processed/stage2_instruction")

# engine_summary_path = STAGE2_ROOT / "engine_comparison_summary.json"
# quality_diag_path = STAGE2_ROOT / "dataset_quality_diagnostics.json"

# if not engine_summary_path.exists():
#     raise FileNotFoundError(f"Missing: {engine_summary_path}")
# if not quality_diag_path.exists():
#     raise FileNotFoundError(f"Missing: {quality_diag_path}")

# with open(engine_summary_path, "r", encoding="utf-8") as f:
#     engine_summary = json.load(f)

# with open(quality_diag_path, "r", encoding="utf-8") as f:
#     quality_diag = json.load(f)

# ranking = engine_summary["variant_summaries"]
# diag_map = {row["variant"]: row for row in quality_diag["diagnostics"]}

# variants = [row["variant"] for row in ranking]
# final_train_losses = [row["final_train_mean_loss"] for row in ranking]
# final_val_losses = [row["final_val_mean_loss"] for row in ranking]
# mean_response_words = [
#     diag_map[v]["overall"]["mean_response_words"] if v in diag_map else None
#     for v in variants
# ]

# plots_dir = STAGE2_ROOT / "report_figures"
# plots_dir.mkdir(parents=True, exist_ok=True)

# def clean_values(values):
#     return [float(v) if v is not None else math.nan for v in values]

# # 1) Final validation loss
# plt.figure(figsize=(8, 5))
# plt.bar(variants, clean_values(final_val_losses))
# plt.title("Final Validation Loss by Engine")
# plt.ylabel("Mean validation loss")
# plt.xlabel("Engine variant")
# plt.xticks(rotation=20)
# plt.tight_layout()
# val_plot_path = plots_dir / "final_validation_loss.png"
# plt.savefig(val_plot_path, dpi=200)
# plt.close()

# # 2) Final train loss
# plt.figure(figsize=(8, 5))
# plt.bar(variants, clean_values(final_train_losses))
# plt.title("Final Training Loss by Engine")
# plt.ylabel("Mean training loss")
# plt.xlabel("Engine variant")
# plt.xticks(rotation=20)
# plt.tight_layout()
# train_plot_path = plots_dir / "final_training_loss.png"
# plt.savefig(train_plot_path, dpi=200)
# plt.close()

# # 3) Mean response length
# plt.figure(figsize=(8, 5))
# plt.bar(variants, clean_values(mean_response_words))
# plt.title("Average Response Length by Engine")
# plt.ylabel("Mean response words")
# plt.xlabel("Engine variant")
# plt.xticks(rotation=20)
# plt.tight_layout()
# resp_plot_path = plots_dir / "mean_response_words.png"
# plt.savefig(resp_plot_path, dpi=200)
# plt.close()

# # 4) Compact markdown summary
# summary_md_path = plots_dir / "engine_results_table.md"
# with open(summary_md_path, "w", encoding="utf-8") as f:
#     f.write("# Engine Results Summary\n\n")
#     f.write("| Variant | Final Train Loss | Final Val Loss | Mean Response Words |\n")
#     f.write("|---|---:|---:|---:|\n")
#     for v, tr, va, rw in zip(variants, final_train_losses, final_val_losses, mean_response_words):
#         tr_str = f"{tr:.4f}" if tr is not None else "NA"
#         va_str = f"{va:.4f}" if va is not None else "NA"
#         rw_str = f"{rw:.2f}" if rw is not None else "NA"
#         f.write(f"| {v} | {tr_str} | {va_str} | {rw_str} |\n")

# print("Saved:")
# print(" -", val_plot_path)
# print(" -", train_plot_path)
# print(" -", resp_plot_path)
# print(" -", summary_md_path)

In [ ]:
# from pathlib import Path
# import json

# if "all_results" not in globals() or not all_results:
#     if not RESULTS_PATH.exists():
#         raise FileNotFoundError(
#             f"Missing results file: {RESULTS_PATH}\n"
#             "Run the quality experiment cell first."
#         )
#     with open(RESULTS_PATH, "r", encoding="utf-8") as f:
#         all_results = json.load(f)
#     print("Loaded all_results from:", RESULTS_PATH)
# else:
#     print("Using in-memory all_results.")

# baseline_variant = BASELINE_VARIANT

# if baseline_variant not in all_results:
#     raise ValueError(
#         f"Baseline variant '{baseline_variant}' not found in all_results. "
#         f"Available: {list(all_results.keys())}"
#     )

# expected_variants = ALL_VARIANTS if "ALL_VARIANTS" in globals() else sorted(all_results.keys())
# missing_variants = [v for v in expected_variants if v not in all_results]

# baseline_train = all_results[baseline_variant].get("final_train_mean_loss", None)
# baseline_val = (all_results[baseline_variant].get("val_result") or {}).get("mean_loss", None)

# comparison_rows = []

# print(f"=== Comparison against baseline: {baseline_variant} ===")
# if missing_variants:
#     print("Missing variants not yet evaluated:", missing_variants)

# for variant in expected_variants:
#     result = all_results.get(variant, {})
#     train_loss = result.get("final_train_mean_loss", None)
#     val_loss = (result.get("val_result") or {}).get("mean_loss", None)

#     train_delta = None if (baseline_train is None or train_loss is None) else (train_loss - baseline_train)
#     val_delta = None if (baseline_val is None or val_loss is None) else (val_loss - baseline_val)

#     if variant == baseline_variant:
#         train_status = "baseline"
#         val_status = "baseline"
#     elif train_loss is None or val_loss is None:
#         train_status = "missing"
#         val_status = "missing"
#     else:
#         train_status = "better" if train_delta < 0 else "worse"
#         val_status = "better" if val_delta < 0 else "worse"

#     comparison_rows.append({
#         "variant": variant,
#         "baseline_variant": baseline_variant,
#         "final_train_mean_loss": train_loss,
#         "final_val_mean_loss": val_loss,
#         "train_delta_vs_baseline": train_delta,
#         "val_delta_vs_baseline": val_delta,
#         "train_status_vs_baseline": train_status,
#         "val_status_vs_baseline": val_status,
#     })

#     train_str = f"{train_loss:.4f}" if train_loss is not None else "missing"
#     val_str = f"{val_loss:.4f}" if val_loss is not None else "missing"
#     train_delta_str = f"{train_delta:+.4f}" if train_delta is not None else "NA"
#     val_delta_str = f"{val_delta:+.4f}" if val_delta is not None else "NA"

#     print(
#         f"{variant}: "
#         f"train={train_str} ({train_delta_str} vs {baseline_variant}) | "
#         f"val={val_str} ({val_delta_str} vs {baseline_variant})"
#     )

# comparison_payload = {
#     "baseline_variant": baseline_variant,
#     "expected_variants": expected_variants,
#     "missing_variants": missing_variants,
#     "rows": comparison_rows,
# }

# comparison_path = STAGE2_ROOT / "baseline_relative_comparison.json"
# with open(comparison_path, "w", encoding="utf-8") as f:
#     json.dump(comparison_payload, f, ensure_ascii=False, indent=2)

# print("\nSaved baseline-relative comparison to:", comparison_path)

In [ ]:
# from pathlib import Path
# import json
# import random
# from collections import defaultdict

# STAGE2_ROOT = Path("/home/CS159FinalProject/data/processed/stage2_instruction")

# EVAL_PACK_SEED = 123
# EVAL_SAMPLES_PER_TASK = 10

# eval_pack_json = STAGE2_ROOT / "heldout_eval_pack.json"
# eval_pack_md = STAGE2_ROOT / "heldout_eval_pack.md"

# # load validation ids from shared split
# split_path = STAGE2_ROOT / "shared_split.json"
# if not split_path.exists():
#     raise FileNotFoundError(f"Missing split file: {split_path}")

# with open(split_path, "r", encoding="utf-8") as f:
#     split_info = json.load(f)

# val_ids = set(split_info["val_image_ids"])
# print("Validation image pool size:", len(val_ids))

# def load_val_rows_by_key(variant):
#     dataset_path = STAGE2_ROOT / variant / "stage2_dataset.jsonl"
#     if not dataset_path.exists():
#         raise FileNotFoundError(f"Missing dataset file: {dataset_path}")

#     rows_by_key = {}
#     with open(dataset_path, "r", encoding="utf-8") as f:
#         for line in f:
#             line = line.strip()
#             if not line:
#                 continue
#             row = json.loads(line)

#             if row["image_id"] not in val_ids:
#                 continue

#             key = (row["image_id"], row["task_type"])
#             rows_by_key[key] = row

#     return rows_by_key

# variant_maps = {variant: load_val_rows_by_key(variant) for variant in ALL_VARIANTS}

# common_keys = None
# for variant in ALL_VARIANTS:
#     keys = set(variant_maps[variant].keys())
#     common_keys = keys if common_keys is None else (common_keys & keys)

# common_keys = sorted(common_keys) if common_keys is not None else []
# print("Common validation keys across all variants:", len(common_keys))

# # keep only instruction-aligned keys
# aligned_keys = []
# for key in common_keys:
#     instructions = [variant_maps[v][key]["instruction"] for v in ALL_VARIANTS]
#     if len(set(instructions)) == 1:
#         aligned_keys.append(key)

# print("Instruction-aligned validation keys:", len(aligned_keys))

# keys_by_task = defaultdict(list)
# for key in aligned_keys:
#     image_id, task_type = key
#     keys_by_task[task_type].append(key)

# rng = random.Random(EVAL_PACK_SEED)

# selected_keys = []
# for task_type, keys in sorted(keys_by_task.items()):
#     keys_copy = keys[:]
#     rng.shuffle(keys_copy)
#     selected_keys.extend(keys_copy[:EVAL_SAMPLES_PER_TASK])

# print("Selected held-out eval examples:", len(selected_keys))

# samples = []
# for key in selected_keys:
#     image_id, task_type = key
#     base_row = variant_maps[ALL_VARIANTS[0]][key]

#     sample = {
#         "image_id": image_id,
#         "task_type": task_type,
#         "instruction": base_row["instruction"],
#         "reference_responses": {},
#     }

#     for variant in ALL_VARIANTS:
#         sample["reference_responses"][variant] = variant_maps[variant][key]["response"]

#     samples.append(sample)

# payload = {
#     "all_variants": ALL_VARIANTS,
#     "num_samples": len(samples),
#     "samples_per_task": EVAL_SAMPLES_PER_TASK,
#     "samples": samples,
# }

# with open(eval_pack_json, "w", encoding="utf-8") as f:
#     json.dump(payload, f, ensure_ascii=False, indent=2)

# with open(eval_pack_md, "w", encoding="utf-8") as f:
#     f.write("# Held-Out Evaluation Pack\n\n")
#     f.write(f"Variants: {', '.join(ALL_VARIANTS)}\n\n")
#     f.write(f"Samples per task: {EVAL_SAMPLES_PER_TASK}\n\n")
#     f.write(f"Total samples: {len(samples)}\n\n")

#     for i, sample in enumerate(samples, start=1):
#         f.write(f"## Eval Sample {i}\n\n")
#         f.write(f"- **image_id:** {sample['image_id']}\n")
#         f.write(f"- **task_type:** {sample['task_type']}\n\n")
#         f.write("**Instruction**\n\n")
#         f.write(sample["instruction"] + "\n\n")

#         for variant in ALL_VARIANTS:
#             f.write(f"### Reference response from {variant}\n\n")
#             f.write(sample["reference_responses"][variant] + "\n\n")

# print("Saved held-out eval pack JSON to:", eval_pack_json)
# print("Saved held-out eval pack markdown to:", eval_pack_md)

In [ ]:
# from pathlib import Path
# import json
# import random

# STAGE2_ROOT = Path("/home/CS159FinalProject/data/processed/stage2_instruction")

# PAIRWISE_JUDGE_SEED = 2026

# eval_pack_json = STAGE2_ROOT / "heldout_eval_pack.json"
# pairwise_requests_json = STAGE2_ROOT / "pairwise_judge_requests.json"
# pairwise_requests_md = STAGE2_ROOT / "pairwise_judge_requests.md"
# pairwise_results_template_json = STAGE2_ROOT / "pairwise_judge_results_template.json"
# pairwise_results_filled_json = STAGE2_ROOT / "pairwise_judge_results_filled.json"
# pairwise_summary_json = STAGE2_ROOT / "pairwise_judge_summary.json"

# if not eval_pack_json.exists():
#     raise FileNotFoundError(
#         f"Missing eval pack: {eval_pack_json}\n"
#         "Run Cell 92 first."
#     )

# with open(eval_pack_json, "r", encoding="utf-8") as f:
#     eval_pack = json.load(f)

# all_variants = eval_pack["all_variants"]
# baseline_variant = BASELINE_VARIANT

# if baseline_variant not in all_variants:
#     raise ValueError(
#         f"BASELINE_VARIANT '{baseline_variant}' not found in eval pack variants: {all_variants}"
#     )

# candidate_variants = [v for v in all_variants if v != baseline_variant]
# if not candidate_variants:
#     raise ValueError("No candidate variants found to compare against the baseline.")

# rng = random.Random(PAIRWISE_JUDGE_SEED)

# def build_judge_prompt(instruction, task_type, assistant_a_text, assistant_b_text):
#     return f"""You are an impartial evaluator for a vision-language instruction-following project.

# Your job is to compare two candidate responses to the SAME instruction.

# Evaluation criteria:
# 1. Accuracy
# 2. Relevance to the instruction
# 3. Helpfulness
# 4. Completeness / level of detail
# 5. Reasoning quality (if the task requires reasoning)

# Task type: {task_type}

# Instruction:
# {instruction}

# Assistant A:
# {assistant_a_text}

# Assistant B:
# {assistant_b_text}

# Choose exactly one winner:
# - assistant_a
# - assistant_b
# - tie

# Return JSON only in this format:
# {{
#   "winner": "assistant_a" or "assistant_b" or "tie",
#   "reason": "one short sentence"
# }}
# """

# judge_requests = []
# results_template = []

# for sample_idx, sample in enumerate(eval_pack["samples"], start=1):
#     image_id = sample["image_id"]
#     task_type = sample["task_type"]
#     instruction = sample["instruction"]
#     ref_map = sample["reference_responses"]

#     baseline_response = ref_map[baseline_variant]

#     for candidate_variant in candidate_variants:
#         candidate_response = ref_map[candidate_variant]

#         pair = [
#             (baseline_variant, baseline_response),
#             (candidate_variant, candidate_response),
#         ]
#         rng.shuffle(pair)

#         (assistant_a_variant, assistant_a_text), (assistant_b_variant, assistant_b_text) = pair

#         request_id = f"{candidate_variant}__sample_{sample_idx:03d}"

#         request = {
#             "request_id": request_id,
#             "image_id": image_id,
#             "task_type": task_type,
#             "instruction": instruction,
#             "baseline_variant": baseline_variant,
#             "candidate_variant": candidate_variant,
#             "assistant_a_variant": assistant_a_variant,
#             "assistant_b_variant": assistant_b_variant,
#             "assistant_a_text": assistant_a_text,
#             "assistant_b_text": assistant_b_text,
#             "judge_prompt": build_judge_prompt(
#                 instruction=instruction,
#                 task_type=task_type,
#                 assistant_a_text=assistant_a_text,
#                 assistant_b_text=assistant_b_text,
#             ),
#         }
#         judge_requests.append(request)

#         results_template.append({
#             "request_id": request_id,
#             "winner": None,   # fill with: assistant_a / assistant_b / tie
#             "reason": "",
#         })

# with open(pairwise_requests_json, "w", encoding="utf-8") as f:
#     json.dump(judge_requests, f, ensure_ascii=False, indent=2)

# with open(pairwise_results_template_json, "w", encoding="utf-8") as f:
#     json.dump(results_template, f, ensure_ascii=False, indent=2)

# with open(pairwise_requests_md, "w", encoding="utf-8") as f:
#     f.write("# Pairwise Judge Requests\n\n")
#     f.write(f"Baseline variant: {baseline_variant}\n\n")
#     f.write(f"Candidate variants: {', '.join(candidate_variants)}\n\n")
#     f.write(f"Total pairwise requests: {len(judge_requests)}\n\n")

#     for req in judge_requests:
#         f.write(f"## {req['request_id']}\n\n")
#         f.write(f"- **image_id:** {req['image_id']}\n")
#         f.write(f"- **task_type:** {req['task_type']}\n")
#         f.write(f"- **baseline_variant:** {req['baseline_variant']}\n")
#         f.write(f"- **candidate_variant:** {req['candidate_variant']}\n")
#         f.write(f"- **assistant_a_variant:** {req['assistant_a_variant']}\n")
#         f.write(f"- **assistant_b_variant:** {req['assistant_b_variant']}\n\n")
#         f.write("**Instruction**\n\n")
#         f.write(req["instruction"] + "\n\n")
#         f.write("### Assistant A\n\n")
#         f.write(req["assistant_a_text"] + "\n\n")
#         f.write("### Assistant B\n\n")
#         f.write(req["assistant_b_text"] + "\n\n")
#         f.write("### Judge Prompt\n\n")
#         f.write("```text\n")
#         f.write(req["judge_prompt"])
#         f.write("\n```\n\n")

# print("Saved pairwise judge requests to:", pairwise_requests_json)
# print("Saved pairwise judge markdown to:", pairwise_requests_md)
# print("Saved results template to:", pairwise_results_template_json)

# # ------------------------------------------------------------
# # Optional summary step:
# # If you later create pairwise_judge_results_filled.json,
# # this block will summarize win-rate automatically.
# # ------------------------------------------------------------
# if pairwise_results_filled_json.exists():
#     with open(pairwise_results_filled_json, "r", encoding="utf-8") as f:
#         filled_results = json.load(f)

#     results_by_id = {row["request_id"]: row for row in filled_results}

#     summary = {
#         "baseline_variant": baseline_variant,
#         "candidate_summaries": [],
#     }

#     for candidate_variant in candidate_variants:
#         wins = 0
#         losses = 0
#         ties = 0
#         missing = 0

#         candidate_requests = [
#             req for req in judge_requests
#             if req["candidate_variant"] == candidate_variant
#         ]

#         for req in candidate_requests:
#             result = results_by_id.get(req["request_id"])
#             if result is None:
#                 missing += 1
#                 continue

#             winner = result.get("winner", None)

#             if winner == "tie":
#                 ties += 1
#                 continue

#             if winner == "assistant_a":
#                 winner_variant = req["assistant_a_variant"]
#             elif winner == "assistant_b":
#                 winner_variant = req["assistant_b_variant"]
#             else:
#                 missing += 1
#                 continue

#             if winner_variant == candidate_variant:
#                 wins += 1
#             elif winner_variant == baseline_variant:
#                 losses += 1
#             else:
#                 missing += 1

#         decided = wins + losses
#         win_rate = wins / decided if decided > 0 else None

#         summary["candidate_summaries"].append({
#             "candidate_variant": candidate_variant,
#             "wins": wins,
#             "losses": losses,
#             "ties": ties,
#             "missing_or_invalid": missing,
#             "decided": decided,
#             "win_rate_over_decided": win_rate,
#         })

#     with open(pairwise_summary_json, "w", encoding="utf-8") as f:
#         json.dump(summary, f, ensure_ascii=False, indent=2)

#     print("\nLoaded filled judge results and saved summary to:", pairwise_summary_json)
#     print(json.dumps(summary, indent=2))
# else:
#     print(
#         "\nNo filled judge results file found yet.\n"
#         f"When ready, create: {pairwise_results_filled_json}\n"
#         "using the same request_id values and winner = assistant_a / assistant_b / tie."
#     )